<a href="https://colab.research.google.com/github/shulruferor-sudo/Advance-Data-Science/blob/main/projects/Bureau%20of%20Statistics/Assignment_Solution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shulruferor-sudo/Advance-Data-Science/blob/main/projects/Bureau%20of%20Statistics/Assignment_Solution.ipynb)


In [26]:
"""
Occupational Classification:
---------------------------

Survey Data and Model Evaluation

This notebook examines survey data and metadata, tests field-specific preparation decisions, and evaluates occupational classification models.

Each transformation and modeling decision is tied to observable evidence and assessed against its alternatives.

"""
import hashlib, io, json, re, pathlib, joblib, importlib, subprocess, sys, warnings, packaging, urllib, urllib3, symbol

# install packages
%pip -q install sentence-transformers==5.1.2 xgboost lightgbm "scikit-learn>=1.6,<1.9"

# loads a pre-trained model for mapping text and sentences into dense vector spaces
from sentence_transformers import SentenceTransformer

# Python part of the warnings subsystem.
warnings.filterwarnings("ignore", category=UserWarning)

# NumPy is collection of fast mathematical operations over arrays
import numpy as np

# SciPy is a collection of mathematical algorithms
from scipy import sparse

# Pandas: organize, inspect, and format survey records as tables.
import pandas as pd

# measure execution time and escape text for HTML reports.
from html import escape
from time import perf_counter

# Matplotlib: create static charts.
import matplotlib.pyplot as plt

# Plotly create interactive charts, combine subplots, and configure rendering
from plotly import express as px, graph_objects as go, io as pio, subplots

# IPython display HTML, Markdown, tables, and other notebook outputs.
from IPython.display import HTML, Markdown, display

# Scikit-learn: feature extraction, preprocessing, modeling, validation, and evaluation.
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder, Normalizer, OneHotEncoder
from sklearn.decomposition import TruncatedSVD
from sklearn.base import BaseEstimator, ClassifierMixin, TransformerMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.utils.validation import check_array, check_is_fitted, check_X_y
from sklearn.model_selection import GridSearchCV, GroupKFold, train_test_split, GroupShuffleSplit, ParameterGrid, StratifiedKFold, cross_val_predict
from sklearn.metrics import accuracy_score, auc, calinski_harabasz_score, classification_report, f1_score, roc_curve
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC, SVC
from sklearn.naive_bayes import ComplementNB
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

# Display interactive charts directly in Colab.
pio.renderers.default = "colab"

# Display every table column and disable mathematical rendering in table HTML.
pd.set_option("display.html.use_mathjax", False)
pd.set_option("display.max_columns", None)


def custom_style(data: pd.DataFrame, key: str = 'running_key'):

    header_properties = [("background-color", "#1e293b"), ("color", "#ffffff"), ("font-family", "sans-serif"), ("font-size", "13px"), ("padding", "10px 15px"), ("text-align", "left")]

    cell_properties = [("font-family", "sans-serif"), ("font-size", "12px"), ("padding", "8px 15px"), ("border-bottom", "1px solid #e2e8f0")]

    styled = (data.set_index(key) if key in data.columns else data).rename_axis(columns=None)\
                                                                   .style.format(precision=2, na_rep="—", escape="html")\
                                                                   .set_table_styles([
                                                                            {"selector": "th", "props": header_properties},
                                                                            {"selector": "td", "props": cell_properties},
                                                                            {"selector": "tr:hover", "props": [("background-color", "#f1f5f9")]}
                                                                          ])

    field_style = ("h3", "color: #1e293b; font-size: 18px; font-weight: 700; border-top: 1px solid #cbd5e1; padding-top: 12px; margin: 20px 0 8px")

    section_style = ("h4", "color: #0f766e; font-size: 14px; font-weight: 700; margin: 16px 0 4px")
    body_style = ("p", "color: #334155; font-size: 13px; margin: 4px 0")
    summary_style = ("p", "color: #475569; font-size: 13px; font-weight: 600; margin: 4px 0 8px")
    empty_style = ("p", "color: #64748b; font-size: 13px; font-style: italic; margin: 4px 0 8px")
    styles = {"field": field_style, "section": section_style, "body": body_style, "summary": summary_style, "empty": empty_style}
    tag, style = styles[role]
    return HTML(f"<{tag} style='font-family: sans-serif; {style}'>{escape(text)}</{tag}>")


# Apply the shared table style whenever a DataFrame is displayed as HTML.
pd.DataFrame._repr_html_ = lambda self: custom_style(self)._repr_html_()

ModuleNotFoundError: No module named 'symbol'

In [ ]:
def transform_to_tfidf(column, feature_fit_rows, norm=None, smooth_idf=False, sublinear_tf=False, use_idf=True):
    """
    Fit TF-IDF on selected records and return one pandas column per learned token.

    Parameters
    ----------
    column : pandas.Series
        A source column with one string per record. Use empty strings for
        missing values so those records produce rows of zeros. The output
        retains the Series index and record order.

    feature_fit_rows : sequence, pandas.Index, slice, or boolean mask
        Selects the records used to learn the vocabulary and IDF weights.
        Selection uses column.loc. Integer labels refer to the Series index.
        A boolean Series must align with that index. An unindexed boolean
        mask must have one entry per record. The selected records must
        collectively contain at least one token.

    norm : {None, "l1", "l2"}, default=None
        Controls normalization after weighting. None preserves the computed
        weights. "l1" scales each nonzero row so its absolute weights sum to
        1. "l2" scales each nonzero row to unit Euclidean length. Rows of
        zeros stay zero.

    smooth_idf : bool, default=False
        Controls inverse document frequency smoothing. False uses
        log(n / df) + 1. True uses log((n + 1) / (df + 1)) + 1.
        Here n is the number of fitting records and df is the number of
        those records containing the token. This option has no effect when
        use_idf is False.

    sublinear_tf : bool, default=False
        Controls how repeated tokens count within a record. False uses the
        occurrence count. True replaces each positive count with
        1 + log(count), reducing the weight of repeated occurrences.

    use_idf : bool, default=True
        Controls whether inverse document frequency weights multiply token
        frequencies. When False, sublinear_tf and norm still apply.

    Returns
    -------
    pandas.DataFrame
        Sparse numeric features with one row per input record and one
        column per learned token. For a named Series, each feature name is
        the source column name, two underscores, and the token, such as
        ShemAvoda__nvidia. An unnamed Series uses the tokens as names.

    Notes
    -----
    The vectorizer lowercases input and includes single-character word
    tokens. Tokens absent from the fitting vocabulary contribute no features.
    """
    column = column.astype("string").fillna("").str.lower().map(lambda value: re.sub(r"[^a-zא-ת ]", "", value))

    vectorizer = TfidfVectorizer(token_pattern=r"(?u)\b\w+\b", norm=norm, smooth_idf=smooth_idf, sublinear_tf=sublinear_tf, use_idf=use_idf)

    # Learn the vocabulary and weights from the fitting records only.
    vectorizer.fit(column.loc[feature_fit_rows])
    tfidf_matrix = vectorizer.transform(column)

    # Keep the source field identifiable when combining feature columns.
    column_prefix = f"{column.name}__" if column.name is not None else ""
    feature_columns = [f"{column_prefix}{token}" for token in vectorizer.get_feature_names_out()]

    return pd.DataFrame.sparse.from_spmatrix(tfidf_matrix, index=column.index, columns=feature_columns)

In [ ]:
model = SentenceTransformer("sentence-transformers/LaBSE", tokenizer_kwargs={"do_lower_case": True}, similarity_fn_name="dot")
model.max_seq_length = 128
model.eval()

sentences = ["This is an example sentence", "Each sentence is converted"]
embeddings = model.encode(sentences, convert_to_numpy=True, show_progress_bar=False)

def transform_column_with_labse(column, alpha=0.5, feature_fit_rows=None):
    """
    Replace values containing Latin letters with their closest Hebrew match when the similarity exceeds alpha.
    """
    values = column.astype("string").str.strip().replace("", pd.NA)
    has_hebrew = values.str.contains(r"[א-ת]", na=False)
    has_latin = values.str.contains(r"[A-Za-z]", na=False)
    latin_cells = values.loc[has_latin]

    # Restrict Hebrew reference values to the fitting records when supplied.
    hebrew_values = values.where(has_hebrew & ~has_latin)
    if feature_fit_rows is not None:
        hebrew_values = hebrew_values.loc[feature_fit_rows]
    hebrew_names = sorted(hebrew_values.dropna().unique())

    transformed_column = column.copy()
    if latin_cells.empty or not hebrew_names:
        return transformed_column

    # Dot products between unit embeddings give cosine similarities.
    vectors = model.encode(latin_cells.tolist() + hebrew_names, batch_size=8, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False)
    latin_count = len(latin_cells)
    similarities = vectors[:latin_count] @ vectors[latin_count:].T

    # Preserve the original value when its best match falls at or below the threshold.
    best_indices = similarities.argmax(axis=1)
    best_scores = similarities[np.arange(latin_count), best_indices]
    best_hebrew = pd.Series(np.asarray(hebrew_names)[best_indices], index=latin_cells.index, dtype="string")
    transformed_column.loc[latin_cells.index] = best_hebrew.where(best_scores > alpha, column.loc[latin_cells.index])

    return transformed_column# Cache the workbook bytes so rerunning the cell can reuse the download.

In [ ]:
# Cache the workbook bytes so rerunning the cell can reuse the download.
if "raw" not in globals() or raw is None:
    URL = "https://raw.githubusercontent.com/shulruferor-sudo/Advance-Data-Science/main/Datasets/cbs_occupation_survey_sample/DUMMY_DATASET.xlsx"
    raw = urllib.request.urlopen(URL, timeout=90).read()

data = pd.read_excel(io.BytesIO(raw), sheet_name="Survey Dataset").dropna(axis=0, how="all").dropna(axis=1, how="all").iloc[:200]
metadata = pd.read_excel(io.BytesIO(raw), sheet_name="Metadata").dropna(axis=0, how="all").dropna(axis=1, how="any")

display(metadata.tail(5))
display(data.tail(2))

def resolve_left_references(data, value, columns):
    """
    Within each row, replace each marker with the nearest populated nonmarker value to its left.
    """
    selected = data.loc[:, columns]
    markers = selected.eq(value).fillna(False)
    blanks = selected.eq("").fillna(False)

    # Scan columns in the supplied order and carry the last usable value across each row.
    nearest_left = selected.mask(markers | blanks).ffill(axis=1)
    resolved = selected.mask(markers & nearest_left.notna(), nearest_left)

    return pd.concat([data.drop(columns=columns), resolved], axis=1).reindex(columns=data.columns)


def manual_explicits_transforms(value):
    """
    Apply the specified text replacements after removing the prefix and surrounding whitespace.
    """
    text_transformations = {"קרית גת": "קריית גת", "לא יכול לפרט": "חסוי", "4 עונות": "ארבע עונות", 'למ"ס': "הלשכה המרכזית לסטטיסטיקה"}
    if isinstance(value, str):
        value = value.removeprefix("א.א.").strip()
    return text_transformations.get(value, value)


def print_free_text_status(column):
    """
    Estimate whether the column is free text by comparing one-time and recurring values.
    """
    counts = column.value_counts()
    display(style_report_text(f"{column.name} is {'a' if counts.eq(1).sum() > counts.gt(1).sum() else 'not a'} free-text feature by this heuristic."))


def print_unique_settlements(column):
    """
    Display the number of distinct nonmissing values.
    """
    display(style_report_text(f"There are {column.nunique()} unique values in {column.name}."))


def print_mandatory_status(column):
    """
    Display whether the field appears mandatory based on observed missing values.
    """
    display(style_report_text(f"{column.name} appears {'optional' if column.isna().any() else 'mandatory'} based on the recorded values."))


def print_workplace_name_counts(column):
    """
    Display counts of missing entries, nonmissing entries, and distinct values.
    """
    display(style_report_text(f"Values are missing from {column.isna().sum():,} records and present in {column.count():,} records, with {column.nunique():,} distinct values overall."))


def group_values_by_script(column):
    """
    Group nonmissing values by the presence of Hebrew and ASCII Latin letters.
    """
    values = column.astype("string")
    has_hebrew = values.str.contains(r"[א-ת]", na=False)
    has_latin = values.str.contains(r"[A-Za-z]", na=False)

    return {"HEBREW ONLY": column.loc[has_hebrew & ~has_latin], "LATIN ONLY": column.loc[has_latin & ~has_hebrew], "MIXED HEBREW + LATIN": column.loc[has_hebrew & has_latin], "NO HEBREW OR LATIN LETTERS": column.loc[column.notna() & ~has_hebrew & ~has_latin]}


def print_script_frequencies(column, key):
    """
    Display value frequencies by script and list the record IDs matched by row index.
    """
    formatted_keys = key.astype("Int64").astype("string").fillna("—")

    for heading, values in group_values_by_script(column).items():
        record_count = len(values)
        value_count = values.nunique()
        section_summary = f"This section contains {record_count:,} {'record' if record_count == 1 else 'records'} and {value_count:,} distinct {'value' if value_count == 1 else 'values'}."
        display(style_report_text(heading, role="section"))
        display(style_report_text(section_summary, role="summary"))

        if values.empty:
            display(style_report_text("No records found.", role="empty"))
            continue

        frequencies = values.value_counts().rename_axis("Value").to_frame("Count")
        frequencies[key.name] = formatted_keys.loc[values.index].groupby(values, sort=False, observed=True).agg(", ".join)
        display(custom_style(frequencies.reset_index(), grouped_id_column=key.name))


for column_name in metadata.loc[metadata["פורמט"].eq("TEXT") & ~metadata["הסבר"].str.contains("סמל", regex=False, na=False), "משתנה"]:
    data[column_name] = data[column_name].astype("string").str.strip()
    column = data[column_name]
    display(style_report_text(column_name, role="field"))
    data = resolve_left_references(data, 'כנ"ל', ["YeshuvAvoda", "ShemAvoda", "SugAvoda", "ShemMachlaka", "SugMachlaka", "EzoAvoda", "TeurPeula", "TeurTafkid"])
    print_free_text_status(column)
    print_unique_settlements(column)
    print_mandatory_status(column)
    print_workplace_name_counts(column)
    print_script_frequencies(column, data["ID"])


data = data.map(manual_explicits_transforms)

data = data.fillna({"MenahelEtMi": 0, "MaamadAvoda": 0, "MakorSachar": 4, "TeudaGvoha": 9})

data["shnotlimud"] = pd.cut(data["shnotlimud"], bins=[6, 12, 14, 16, 20, 24, 100], include_lowest=True).cat.codes.astype(float)

data["Gil"] = pd.cut(data.Gil, bins=[18, 25, 35, 45, 55, 67, 120], include_lowest=True).cat.codes.astype(float)

data = pd.concat([data.drop(columns=["YeshuvAvoda"]), pd.get_dummies(data["YeshuvAvoda"].fillna("empty"), prefix="YeshuvAvoda", dtype=float)], axis=1)

data["shanimOved"] = (pd.Timestamp.now(tz="UTC") - pd.to_datetime(data.TarichHathala, utc=True)).dt.days / 365.25

for column_name in ["ShemAvoda", "SugMachlaka", "EzoAvoda", "TeurPeula"]:
    data = pd.concat([data.drop(columns=column_name), transform_column_with_labse(data[column_name], feature_fit_rows=report_fit_rows.reindex(data.index, fill_value=False))], axis=1).reindex(columns=data.columns)

for column_name in ["ShemAvoda", "SugAvoda", "ShemMachlaka", "SugMachlaka", "EzoAvoda", "TeurPeula", "TeurTafkid"]:
    data = pd.concat([data.drop(columns=column_name), transform_to_tfidf(data[column_name], report_fit_rows.reindex(data.index, fill_value=False))], axis=1)

print("Final data shape shape:", data.shape)

In [ ]:
data = data.set_index("ID")
data.index.name = "ID"

pd.DataFrame.row_pop = lambda self, idx: (r := self.loc[idx], self.drop(idx, inplace=True))[0]

data = data.row_pop[data.SemelMishlachSofi.fullmatch(r"\d+")]

targets = data.pop("SemelMishlachSofi")

train_data, test_data = train_test_split(data, test_size=30, random_state=42)
train_data, validation_data = train_test_split(train_data, test_size=30, random_state=42)




In [ ]:
class PlotGallery:
    """
    Collect named plots and display them with their explanations.
    """

    def __init__(self):
        self.plots = {}

    def add(self, name, figure, guide="", details=()):
        # Apply shared readable defaults once for every chart.
        if isinstance(figure, go.Figure):
            figure.update_layout(template="plotly_white", font_size=16, title_font_size=21, hoverlabel_font_size=16)
            if not figure.layout.title.text:
                figure.update_layout(title_text=name)
            if figure.layout.height is None:
                figure.update_layout(height=440)
            figure.update_xaxes(tickfont_size=15, automargin=True)
            figure.update_yaxes(tickfont_size=15, automargin=True)
            figure.update_annotations(font_size=17)
        else:
            # Keep Matplotlib figures from displaying before the gallery runs.
            plt.close(figure)
        # Reusing a name replaces that plot without adding a duplicate.
        self.plots[name] = (figure, guide, tuple(details))

    def show(self, *names):
        # With no selection, preserve the order in which plots were added.
        display(Markdown("# Assignment results\n\n## Predict the CBS-assigned occupation code from the participant's online questionnaire answers.\n\n### Participants supply the questionnaire answers. CBS staff assign the standardized code columns. TF-IDF and one-hot encoding describe how selected fields are prepared."))
        for name in names or self.plots:
            figure, guide, details = self.plots[name]
            # H1 names the graph, H2 explains its purpose, and H3 gives the reading key.
            paragraphs = [line.strip().lstrip("- ") for line in guide.splitlines() if line.strip() and not line.lstrip().startswith("#")]
            explanation = "\n\n".join(("## " if index == 0 else "### ") + paragraph for index, paragraph in enumerate(paragraphs))
            display(Markdown("# " + name + "\n\n" + explanation))
            for detail in details:
                display(detail)
            if isinstance(figure, go.Figure):
                figure.show()
            else:
                display(figure)

    def clear(self):
        self.plots.clear()

    def matrices(self, covariance, correlation, pair_counts, field_labels, guide, details=()):
        """Queue side-by-side 2D and rotating 3D views of two recorded-value matrices."""
        fields = covariance.columns.tolist()
        axis_labels = [field_labels[field] for field in fields]
        titles = ("Covariance of recorded codes", "Pearson correlation of recorded codes")

        # Use the same field pairs and sample counts in both views.
        heatmaps = make_subplots(rows=1, cols=2, subplot_titles=titles, horizontal_spacing=0.19)
        three_d = make_subplots(rows=1, cols=2, specs=[[{"type": "scene"}, {"type": "scene"}]], subplot_titles=titles)
        colors = [[0, "#b64a46"], [0.5, "#f7f7f2"], [1, "#32865d"]]
        covariance_limit = float(np.nanmax(np.abs(covariance.to_numpy())))
        views = [(covariance, covariance_limit, "Covariance"), (correlation, 1, "Correlation")]
        x = np.tile(fields, len(fields))
        y = np.repeat(fields, len(fields))
        counts = pair_counts.to_numpy().ravel()
        hover = "%{y} × %{x}<br>Value: %{z:.3f}<br>Recorded pairs: %{customdata}<extra></extra>"

        # Each 2D cell and 3D point represents one measured field pair.
        for column, (matrix, limit, title) in enumerate(views, start=1):
            values = matrix.to_numpy()
            heatmaps.add_trace(go.Heatmap(x=fields, y=fields, z=values, texttemplate="%{z:.2f}", textfont={"size": 19, "color": "#152b26"}, customdata=pair_counts.to_numpy(), colorscale=colors, zmin=-limit, zmax=limit, xgap=3, ygap=3, hoverongaps=False, colorbar={"title": title, "x": 0.42 if column == 1 else 1.02, "len": 0.72}, hovertemplate=hover), row=1, col=column)
            heatmaps.update_xaxes(tickmode="array", tickvals=fields, ticktext=axis_labels, tickangle=-35, automargin=True, row=1, col=column)
            heatmaps.update_yaxes(tickmode="array", tickvals=fields, ticktext=axis_labels, autorange="reversed", automargin=True, row=1, col=column)

            # Stems connect zero to the actual matrix values without interpolating categories.
            values = values.ravel()
            recorded = np.isfinite(values)
            stem_x = [point for value in x[recorded] for point in (value, value, None)]
            stem_y = [point for value in y[recorded] for point in (value, value, None)]
            stem_z = [point for value in values[recorded] for point in (0, value, None)]
            three_d.add_trace(go.Scatter3d(x=stem_x, y=stem_y, z=stem_z, mode="lines", line={"color": "#718d81", "width": 5}, hoverinfo="skip", showlegend=False), row=1, col=column)
            three_d.add_trace(go.Scatter3d(x=x[recorded], y=y[recorded], z=values[recorded], mode="markers", marker={"size": 7, "opacity": 0.9, "color": values[recorded], "colorscale": colors, "cmin": -limit, "cmax": limit, "colorbar": {"title": title, "x": 0.46 if column == 1 else 1.02, "len": 0.65}}, customdata=counts[recorded], hovertemplate=hover, showlegend=False), row=1, col=column)

        # Preserve room for full labels and place the pair counts with the heatmaps.
        heatmaps.update_layout(height=820, margin={"t": 120, "b": 160})
        self.add("Relationships between the four survey fields", heatmaps, guide, details=details)

        # Animate only camera positions; matrix values stay unchanged.
        axis = {"type": "category", "categoryorder": "array", "categoryarray": fields, "tickmode": "array", "tickvals": fields, "ticktext": axis_labels, "tickfont": {"size": 13}}
        angles = np.linspace(np.pi / 4, 2 * np.pi + np.pi / 4, 24, endpoint=False)
        eyes = [{"x": float(1.8 * np.cos(angle)), "y": float(1.8 * np.sin(angle)), "z": 0.9} for angle in angles]
        three_d.frames = [go.Frame(name=f"camera_{number}", layout={"scene": {"camera": {"eye": eye}}, "scene2": {"camera": {"eye": eye}}}) for number, eye in enumerate(eyes)]
        buttons = [{"label": "Play rotation", "method": "animate", "args": [None, {"frame": {"duration": 150, "redraw": True}, "transition": {"duration": 0}, "fromcurrent": True}]}, {"label": "Pause", "method": "animate", "args": [[None], {"frame": {"duration": 0, "redraw": False}, "transition": {"duration": 0}, "mode": "immediate"}]}]
        three_d.update_scenes(xaxis=axis, yaxis=axis, camera_eye=eyes[0])
        three_d.update_layout(height=900, margin={"t": 140, "b": 50}, scene={"zaxis_title": "Covariance"}, scene2={"zaxis_title": "Correlation"}, updatemenus=[{"type": "buttons", "direction": "left", "x": 0, "y": 1.06, "buttons": buttons}])
        self.add("Four-field matrices in 3D", three_d, "View the same covariance and correlation values in 3D.\nGreen means positive, red means negative. Height and color show the recorded numerical value, not model accuracy.\nDrag or press Play rotation. Hover shows the number of recorded pairs.")
        return heatmaps, three_d

    def diagnostic(self, name, datasets, models, series, x_title, z_title, guide='', details=(), reference=None):
        """Queue a 3D diagnostic with a separate filled lane for each classifier."""
        figure = go.Figure()
        trace_datasets = []

        # Each series supplies actual x and z points for one classifier and preparation.
        for curve in series:
            x = np.asarray(curve['x'], dtype=float)
            z = np.asarray(curve['z'], dtype=float)
            lane = models.index(curve['model'])
            color = px.colors.sample_colorscale([[0, '#d73c46'], [1, '#329b5a']], [float(np.clip(curve['score'], 0, 1))])[0]
            visible = curve['dataset'] == datasets[0]

            # Fill from zero to the curve within this lane, with no connections between classifiers.
            wall_x = np.vstack([x, x])
            wall_y = np.full_like(wall_x, lane)
            wall_z = np.vstack([np.zeros_like(z), z])
            figure.add_trace(go.Surface(x=wall_x, y=wall_y, z=wall_z, colorscale=[[0, color], [1, color]], opacity=0.35, showscale=False, hoverinfo='skip', showlegend=False, visible=visible))

            # The visible line keeps the exact values and hover details.
            figure.add_trace(go.Scatter3d(x=x, y=np.full(len(x), lane), z=z, mode='lines+markers', name=curve['model'], line={'color': color, 'width': 6}, marker={'color': color, 'size': 3}, customdata=curve['hover_data'], hovertemplate=curve['hover'], showlegend=True, visible=visible))
            trace_datasets.extend([curve['dataset'], curve['dataset']])

        # A diagonal reference is shown beside the classifier lanes when relevant.
        if reference is not None:
            figure.add_trace(go.Scatter3d(x=[0, 1], y=[-0.7, -0.7], z=[0, 1], mode='lines', name=reference, line={'color': '#64748b', 'width': 5, 'dash': 'dash'}, hovertemplate=reference + '<extra></extra>'))

        # The menu changes visibility only; the underlying validation scores stay fixed.
        buttons = []
        for dataset in datasets:
            visible = [name == dataset for name in trace_datasets]
            if reference is not None:
                visible.append(True)
            buttons.append({'label': dataset, 'method': 'update', 'args': [{'visible': visible}]})

        # Label every classifier lane and keep the plot compact in Colab.
        scene = {'xaxis': {'title': x_title}, 'yaxis': {'title': 'Classifier', 'tickmode': 'array', 'tickvals': list(range(len(models))), 'ticktext': models, 'range': [-1, len(models)]}, 'zaxis': {'title': z_title, 'range': [0, 1.03]}, 'camera': {'eye': {'x': 1.45, 'y': -1.65, 'z': 1.05}}}
        figure.update_layout(height=590, margin={'t': 90, 'b': 20, 'l': 5, 'r': 270}, scene=scene, legend={'title': {'text': 'Classifier'}, 'font': {'size': 15}, 'x': 1, 'xanchor': 'left', 'y': 1, 'yanchor': 'top'}, updatemenus=[{'buttons': buttons, 'font': {'size': 12}, 'x': 0, 'y': 1.10, 'xanchor': 'left', 'yanchor': 'top'}])
        self.add(name, figure, guide=guide, details=details)
        return figure

    def diagnostics(self, diagnostic_datasets, diagnostic_models, roc_summary, roc_curves, false_positive_grid, calibration_results, diagnostic_fold_results, diagnostic_selected_dataset, diagnostic_selected_model, roc_coverage):
        """Queue ROC, reliability and fold variation from the same validation results."""
        roc_series = []
        calibration_series = []
        stability_series = []

        # Assemble the actual plotted values, grouped by feature preparation and classifier.
        for dataset in diagnostic_datasets:
            for row in roc_summary.loc[roc_summary['Feature preparation'].eq(dataset)].to_dict('records'):
                pair = (dataset, row['Classifier'])
                if pair not in roc_curves:
                    continue
                hover_data = np.tile([row['Mean ROC AUC'], row['Minimum ROC-evaluable codes'], row['Maximum ROC-evaluable codes']], (len(false_positive_grid), 1))
                hover = 'False-positive rate %{x:.1%}<br>True-positive rate %{z:.1%}<br>Mean ROC AUC %{customdata[0]:.3f}<br>Evaluable codes %{customdata[1]:.0f}–%{customdata[2]:.0f}<extra>%{fullData.name}</extra>'
                roc_series.append({'dataset': dataset, 'model': row['Classifier'], 'x': false_positive_grid, 'z': roc_curves[pair], 'score': row['Mean ROC AUC'], 'hover_data': hover_data, 'hover': hover})

            for model in diagnostic_models:
                # Confidence bands compare the model's probability with observed correctness.
                points = calibration_results.loc[calibration_results['Feature preparation'].eq(dataset) & calibration_results['Classifier'].eq(model)]
                if not points.empty:
                    accuracy = np.average(points['Observed accuracy'], weights=points['Records'])
                    hover = 'Mean confidence %{x:.1%}<br>Observed accuracy %{z:.1%}<br>Records %{customdata[0]}<extra>%{fullData.name}</extra>'
                    calibration_series.append({'dataset': dataset, 'model': model, 'x': points['Mean confidence'], 'z': points['Observed accuracy'], 'score': accuracy, 'hover_data': points[['Records']], 'hover': hover})

                # Fold scores show how performance varies across validation splits.
                rounds = diagnostic_fold_results.loc[diagnostic_fold_results['Feature preparation'].eq(dataset) & diagnostic_fold_results['Classifier'].eq(model)]
                if not rounds.empty:
                    hover = 'Round %{x:.0f}<br>Macro F1 %{z:.3f}<br>Accuracy %{customdata[0]:.1%}<br>Unseen-code records %{customdata[1]}<extra>%{fullData.name}</extra>'
                    stability_series.append({'dataset': dataset, 'model': model, 'x': rounds['Round'], 'z': rounds['Macro F1'], 'score': rounds['Macro F1'].mean(), 'hover_data': rounds[['Accuracy', 'Records with codes absent from fitting']], 'hover': hover})

        # Describe each plot and insert the selected classifier's actual results.
        selected = lambda table: table.loc[table['Feature preparation'].eq(diagnostic_selected_dataset) & table['Classifier'].eq(diagnostic_selected_model)]
        selected_roc = selected(roc_summary)
        selected_bands = selected(calibration_results)
        selected_rounds = selected(diagnostic_fold_results)

        roc_guide = 'ROC shows how well scores distinguish occupation codes.\n\nA curve near the top-left is stronger than one near the diagonal; high AUC can coexist with low exact-code F1. Red means lower AUC, green higher.'
        if not selected_roc.empty:
            result = selected_roc.iloc[0]
            roc_guide += f"\n\nThis run: {diagnostic_selected_model} has mean AUC {result['Mean ROC AUC']:.2f}, with {int(result['Minimum ROC-evaluable codes'])}–{int(result['Maximum ROC-evaluable codes'])} evaluable codes per round."
        roc_guide += ' Feature preparation preceded the split, so this is exploratory.'

        calibration_guide = 'Reliability compares predicted confidence with observed accuracy.\n\nBelow the diagonal indicates overconfidence; above it indicates underconfidence. Colors show accuracy, not calibration quality.'
        if not selected_bands.empty:
            result = selected_bands.sort_values('Records', ascending=False, kind='stable').iloc[0]
            calibration_guide += f"\n\nThis run: {diagnostic_selected_model}'s largest band contains {int(result['Records'])} predictions at {result['Mean confidence']:.0%} mean confidence and {result['Observed accuracy']:.0%} accuracy."

        stability_guide = 'Fold variation shows how macro F1 changes across validation splits.\n\nLarger changes indicate less stable results across these splits. Red means lower mean F1, green higher.'
        if not selected_rounds.empty:
            scores = selected_rounds['Macro F1']
            stability_guide += f'\n\nThis run: {diagnostic_selected_model} ranges from {scores.min():.2f} to {scores.max():.2f} macro F1.'

        # Queue all three views; the final gallery displays them together.
        roc = self.diagnostic('Validation ROC', diagnostic_datasets, diagnostic_models, roc_series, 'False-positive rate', 'True-positive rate', guide=roc_guide, details=(roc_coverage,), reference='Chance reference')
        reliability = self.diagnostic('Probability reliability', diagnostic_datasets, diagnostic_models, calibration_series, 'Mean confidence', 'Observed accuracy', guide=calibration_guide, reference='Confidence equals accuracy')
        stability = self.diagnostic('Validation fold variation', diagnostic_datasets, diagnostic_models, stability_series, 'Validation round', 'Macro F1', guide=stability_guide)
        return roc, reliability, stability


# Start a new collection for this complete notebook run.
plots = PlotGallery()


In [ ]:
# Examine the four recorded survey fields together, before their model imputations.
survey_fields = ["MenahelEtMi", "MaamadAvoda", "MakorSachar", "TeudaGvoha"]
survey_answers = data[survey_fields].apply(pd.to_numeric, errors="coerce")

# Use the questionnaire's field names in the explanations and chart axes.
field_labels = {"MenahelEtMi": "Management", "MaamadAvoda": "Employment status", "MakorSachar": "Salary source", "TeudaGvoha": "Education"}
field_axis_labels = [field_labels[field] for field in survey_fields]


# Compare the recorded response distribution of the four survey fields.
# The same numeric code can have a different meaning in each field.
response_meanings = {
    "MenahelEtMi": {"1": "Manages employees", "2": "Manages managers", "3": "Manages employees and managers"},
    "MaamadAvoda": {"1": "Employee, including a salaried company owner", "2": "Self-employed, employs 1–2 workers", "3": "Self-employed, employs 3 or more workers", "4": "Self-employed, employs no workers", "5": "Cooperative member", "6": "Kibbutz member", "7": "Unpaid family worker"},
    "MakorSachar": {"1": "Workplace", "2": "Contractor company", "3": "Employment agency", "4": "Other"},
    "TeudaGvoha": {"1": "Primary or middle school certificate", "2": "Secondary school certificate without matriculation", "3": "Matriculation certificate", "4": "Non-academic post-secondary certificate", "5": "Bachelor's degree or equivalent", "6": "Master's degree or equivalent, including medical doctorate", "7": "Doctoral degree or equivalent", "8": "Other certificate", "9": "No certificate"},
}

# Count actual recorded codes and missing answers for each field.
response_rows = []
for field in survey_fields:
    counts = survey_answers[field].value_counts(dropna=False)
    for answer, count in counts.items():
        code = "Not recorded" if pd.isna(answer) else str(int(answer)) if float(answer).is_integer() else str(answer)
        meaning = "Not recorded" if pd.isna(answer) else response_meanings[field].get(code, f"Unlisted code {code}")
        response_rows.append({"Field": field, "Answer code": code, "Meaning": meaning, "Records": int(count), "Percentage": count / len(survey_answers), "Label": str(int(count)) if count >= 8 else ""})

survey_response_counts = pd.DataFrame(response_rows)
response_colors = {"1": "#327c89", "2": "#d58d48", "3": "#5873a8", "4": "#9471a9", "5": "#b66569", "6": "#64a885", "7": "#9f8c5a", "8": "#b46d9c", "9": "#699eaf", "Not recorded": "#aeb8bf"}

# Explain the codes using a compact table shown with the chart.
management_recorded = survey_answers["MenahelEtMi"].count()
large_table_styles = [{"selector": "th, td", "props": [("font-size", "16px"), ("padding", "10px")]}]
response_key = pd.DataFrame([{"Field": f"{field_labels[field]} ({field})", "Response code and meaning": "; ".join(f"{code}: {meaning}" for code, meaning in response_meanings[field].items())} for field in survey_fields])
response_key_style = custom_style(response_key).set_properties(subset=["Response code and meaning"], **{"white-space": "normal", "max-width": "64em"}).set_table_styles(large_table_styles, overwrite=False)

# Count answers without changing the original data.
distribution_figure = px.bar(survey_response_counts, x="Records", y="Field", color="Answer code", orientation="h", text="Label", barmode="stack", color_discrete_map=response_colors, category_orders={"Field": survey_fields, "Answer code": [str(code) for code in range(1, 10)] + ["Not recorded"]}, hover_data={"Meaning": True, "Records": True, "Percentage": ":.1%", "Label": False}, title="Recorded responses across the four survey fields")
distribution_figure.update_traces(texttemplate="%{text}", textposition="inside", insidetextanchor="middle", textfont={"size": 17, "color": "white"})
distribution_figure.update_yaxes(autorange="reversed", title=None, tickvals=survey_fields, ticktext=field_axis_labels, automargin=True)
distribution_figure.update_xaxes(title="Number of records", rangemode="tozero")
distribution_figure.update_layout(height=620, legend_title="Answer code", legend={"orientation": "v", "x": 1.02, "y": 1, "font": {"size": 15}}, margin={"t": 100, "b": 80, "r": 180})

# Queue the distribution and its meaning key for the final gallery.
distribution_guide = f"Each bar shows all records. Gray means no recorded answer, including questions that may not apply. Management is recorded for {management_recorded:,} of {len(survey_answers):,} records. Codes have different meanings in each field; use the table and hover for counts."
plots.add("Recorded responses across the four survey fields", distribution_figure, distribution_guide, details=(response_key_style,))

# Calculate each pair using records where both answers are present.
field_covariance = survey_answers.cov()
field_correlation = survey_answers.corr(method="pearson")
field_recorded = survey_answers.notna().astype(int)
field_pair_counts = field_recorded.T.dot(field_recorded)
pair_count_table = custom_style(field_pair_counts).set_table_styles(large_table_styles, overwrite=False)

# Queue the heatmaps and rotating 3D views of the same observed values.
matrix_guide = f"Inspect numerical relationships between the four survey fields.\nGreen: positive association. Red: negative association. Pale: near zero. Stronger color means greater magnitude within that panel.\nThese compare category codes. The management pairs use at most {management_recorded:,} records. Use classification results to assess whether the fields improve prediction."
matrix_figure, matrix_3d_figure = plots.matrices(field_covariance, field_correlation, field_pair_counts, field_labels, matrix_guide, details=(pair_count_table,))


In [ ]:
# Adapt Or Shulrufer's balanced-subset ensemble to the occupation-code target.
class OrBalancedOccupationEnsemble(ClassifierMixin, BaseEstimator):
    """
    Average four classifier families trained on balanced multiclass subsets.

    samples_per_class limits each class's contribution to one subset. Classes
    with fewer records keep every record. Larger classes use successive chunks
    of one seeded shuffle, cycling after their available chunks are exhausted.
    Every subset contains every class found in the supplied fitting records.

    n_subsets controls the number of balanced fitting subsets. Every subset
    fits fresh XGBoost, Random Forest, LightGBM and Complement Naive Bayes models.
    max_depth controls all three tree families together. naive_bayes_alpha
    controls the Naive Bayes smoothing. GridSearchCV selects these two settings
    jointly from four combinations by the complete ensemble's validation F1.

    Source: https://github.com/OrShulrufer/Ensembles
    This is a multiclass adaptation of the original binary implementation.
    It retains balanced subsets and probability averaging. A binary threshold
    is replaced by the highest-scoring occupation code. The small per-code
    counts do not support a separate calibration partition for the full vote.
    """

    def __init__(self, samples_per_class=3, n_subsets=3, max_depth=6, naive_bayes_alpha=1, random_state=42):
        self.samples_per_class = samples_per_class
        self.n_subsets = n_subsets
        self.max_depth = max_depth
        self.naive_bayes_alpha = naive_bayes_alpha
        self.random_state = random_state

    def fit(self, X, y):
        self.__dict__.pop("models_", None)
        if not isinstance(self.samples_per_class, int) or self.samples_per_class < 1:
            raise ValueError("samples_per_class must be a positive integer.")
        if not isinstance(self.n_subsets, int) or self.n_subsets < 1:
            raise ValueError("n_subsets must be a positive integer.")

        X, y = check_X_y(X, y, accept_sparse="csr")
        self.label_encoder_ = LabelEncoder().fit(y)
        self.classes_ = self.label_encoder_.classes_
        self.n_features_in_ = X.shape[1]
        encoded_targets = self.label_encoder_.transform(y)
        generator = np.random.default_rng(self.random_state)
        class_chunks = []

        for code in range(len(self.classes_)):
            rows = generator.permutation(np.flatnonzero(encoded_targets == code))
            chunks = [rows[start:start + self.samples_per_class] for start in range(0, len(rows), self.samples_per_class)]
            class_chunks.append(chunks)

        fitted_models = []
        subset_summary = []
        for subset in range(self.n_subsets):
            selected_rows = np.concatenate([chunks[subset % len(chunks)] for chunks in class_chunks])
            selected_rows = generator.permutation(selected_rows)
            subset_targets = encoded_targets[selected_rows]
            subset_counts = np.bincount(subset_targets, minlength=len(self.classes_))
            subset_summary.append({"Subset": subset + 1, "Records": len(selected_rows), "Classes": len(self.classes_), "Minimum per class": int(subset_counts.min()), "Maximum per class": int(subset_counts.max())})

            # Use fresh members for each subset and preserve local class encoding.
            member_seed = self.random_state + subset
            members = [
                ("XGBoost", XGBClassifier(n_estimators=40, max_depth=self.max_depth, learning_rate=0.1, eval_metric="mlogloss", tree_method="hist", random_state=member_seed, n_jobs=1)),
                ("Random forest", RandomForestClassifier(n_estimators=80, max_depth=self.max_depth, random_state=member_seed, n_jobs=1)),
                ("LightGBM", LGBMClassifier(n_estimators=40, max_depth=self.max_depth, num_leaves=min(15, 2 ** self.max_depth), min_child_samples=2, learning_rate=0.1, verbosity=-1, random_state=member_seed, n_jobs=1)),
                ("Complement Naive Bayes", ComplementNB(alpha=self.naive_bayes_alpha)),
            ]
            for name, estimator in members:
                estimator.fit(X[selected_rows], subset_targets)
                fitted_models.append((f"{name} {subset + 1}", estimator))

        self.models_ = fitted_models
        self.subset_summary_ = pd.DataFrame(subset_summary)
        return self

    def predict_proba(self, X):
        check_is_fitted(self, "models_")
        X = check_array(X, accept_sparse="csr")
        if X.shape[1] != self.n_features_in_:
            raise ValueError("The number of input features differs from the fitting data.")

        probabilities = np.zeros((X.shape[0], len(self.classes_)))
        for _, estimator in self.models_:
            member_codes = np.asarray(estimator.classes_, dtype=int)
            probabilities[:, member_codes] += estimator.predict_proba(X) / len(self.models_)
        return probabilities

    def predict(self, X):
        probabilities = self.predict_proba(X)
        return self.classes_[probabilities.argmax(axis=1)]




# XGBoost requires consecutive integer labels. Keep occupation codes at the interface.
class OccupationXGBoost(ClassifierMixin, BaseEstimator):
    """Encode occupation labels for XGBoost and return the original codes."""

    def __init__(self, random_state=42):
        self.random_state = random_state

    def fit(self, X, y):
        self.label_encoder_ = LabelEncoder().fit(y)
        self.classes_ = self.label_encoder_.classes_
        self.model_ = XGBClassifier(n_estimators=40, max_depth=2, learning_rate=0.1, eval_metric="mlogloss", tree_method="hist", random_state=self.random_state, n_jobs=1)
        self.model_.fit(X, self.label_encoder_.transform(y))
        return self

    def predict_proba(self, X):
        return self.model_.predict_proba(X)

    def predict(self, X):
        return self.classes_[self.predict_proba(X).argmax(axis=1)]


# Compare exactly the four requested classifiers and Or Shulrufer's ensemble.
# The ensemble has four joint combinations, not separately optimized members.
model_families = {
    "XGBoost": (OccupationXGBoost(random_state=SEED), {}),

    "Random forest": (RandomForestClassifier(n_estimators=80, class_weight="balanced", random_state=SEED, n_jobs=1), {"classifier__min_samples_leaf": [1, 2]}),

    "LightGBM": (LGBMClassifier(n_estimators=40, num_leaves=7, min_child_samples=2, learning_rate=0.1, verbosity=-1, random_state=SEED, n_jobs=1), {}),

    "Complement Naive Bayes": (ComplementNB(alpha=1), {}),

    "Or balanced occupation ensemble": (OrBalancedOccupationEnsemble(random_state=SEED), {"classifier__max_depth": [3, 6], "classifier__naive_bayes_alpha": [0.5, 1]}),
}


class DigitExclusiveSelector(TransformerMixin, BaseEstimator):

    def __init__(self, min_support=3):
        self.min_support = min_support

    def fit(self, X, y):
        values = sparse.csc_matrix(X)
        targets = np.asarray(y, dtype=str)
        digit_codes = np.asarray([[code[position] for position in range(4)] for code in targets])

        self.support_ = np.diff(values.indptr)
        self.exclusive_ = np.zeros(values.shape[1], dtype=bool)

        for column in np.flatnonzero(self.support_ >= self.min_support):
            rows = values.indices[values.indptr[column]:values.indptr[column + 1]]
            self.exclusive_[column] = any(np.unique(digit_codes[rows, position]).size == 1 for position in range(4))

        self.mask_ = self.exclusive_.copy()

        if not self.mask_.any():
            self.mask_[:] = True

        return self

    def transform(self, X):

        return X[:, self.mask_]

base_feature_name = "For seven text fields: clean the text, use LaBSE matching on four fields, then apply TF-IDF. Separately: one-hot encode categorical fields, including workplace settlement, and prepare numeric fields. Combine the results"
# Describe each combination of optional steps in the order it runs.
preparation_variants = []

for l2, select, svd in product([False, True], repeat=3):
    preparation_steps = [base_feature_name]

    # L2 normalization applies to the combined feature vector.
    if l2:
        preparation_steps.append("Then apply L2 normalization to the combined features")

    # Digit selection follows normalization when it is enabled.
    if select:
        preparation_steps.append("Then select digit-aligned features")

    # SVD is the final optional transformation.
    if svd:
        preparation_steps.append("Then apply truncated SVD with 16 components")

    preparation_variants.append({"Dataset": ". ".join(preparation_steps), "L2": l2, "Digit selection": select, "SVD": svd})

dataset_options = pd.DataFrame(preparation_variants)


# Fit each representation and classifier within the same grouped folds.
parameter_grids = []
candidate_descriptions = []

for dataset in dataset_options.to_dict("records"):
    for family, (classifier, parameters) in model_families.items():
        if dataset["SVD"] and family in {"Complement Naive Bayes", "Or balanced occupation ensemble"}:
            continue

        parameter_grids.append({"normalize": [Normalizer() if dataset["L2"] else "passthrough"], "select": [DigitExclusiveSelector(min_support=3) if dataset["Digit selection"] else "passthrough"], "svd": [TruncatedSVD(n_components=16, random_state=SEED) if dataset["SVD"] else "passthrough"], "classifier": [classifier], **parameters})
        candidate_descriptions.extend([{"Dataset": dataset["Dataset"], "Classifier": family, "L2": dataset["L2"], "Digit selection": dataset["Digit selection"], "SVD": dataset["SVD"], "Parameters": json.dumps(combination)} for combination in ParameterGrid(parameters)])

model_pipeline = Pipeline([("normalize", "passthrough"), ("select", "passthrough"), ("svd", "passthrough"), ("classifier", RandomForestClassifier())])
# Score each fitted candidate once and retain it for ROC and learning-curve endpoints.
# The key identifies its complete parameter setting and its validation records.
validation_fit_cache = {}


def evaluate_classifier_once(estimator, features, targets):
    """Save the fitted validation model and return the two model-selection scores."""
    cache_key = (joblib.hash(clone(estimator)), tuple(targets.index))
    validation_fit_cache[cache_key] = estimator
    predictions = estimator.predict(features)
    return {"macro_f1": f1_score(targets, predictions, average="macro", zero_division=0), "accuracy": accuracy_score(targets, predictions)}


search = GridSearchCV(model_pipeline, parameter_grids, cv=validation_splits, scoring=evaluate_classifier_once, refit="macro_f1", n_jobs=3, error_score="raise")

# Model selection uses only development records. The 30 test records stay outside it.
# Threading shares the cache, and each member model uses one CPU worker.
print(f"Model selection: {len(candidate_descriptions)} joint configurations × {len(validation_splits)} validation rounds.")
comparison_started = perf_counter()
with joblib.parallel_backend("threading"):
    search.fit(development_matrix, development_targets)

scores = pd.DataFrame(search.cv_results_)[["mean_test_macro_f1", "std_test_macro_f1", "mean_test_accuracy"]]
model_results = pd.concat([pd.DataFrame(candidate_descriptions), scores], axis=1)
best_configuration = model_results.loc[search.best_index_]
ranked_models = model_results.sort_values("mean_test_macro_f1", ascending=False, kind="stable").drop_duplicates(["Dataset", "Classifier"])
ranked_models = ranked_models.rename(columns={"Classifier": "Model", "mean_test_macro_f1": "Macro F1", "std_test_macro_f1": "Fold SD", "mean_test_accuracy": "Accuracy"})

print(f"{len(dataset_options)} representations, {len(model_results)} configurations, {len(validation_splits)} grouped folds.")
print("Exploratory scores. The text features were prepared before the split.")
# Rank the displayed configurations by their mean validation macro-F1.
ranked_display = ranked_models.sort_values("Macro F1", ascending=False, kind="stable").head(18)
ranked_display = ranked_display[["Dataset", "Model", "Macro F1", "Accuracy"]].rename(columns={"Dataset": "Feature pipeline", "Model": "Classifier"}).reset_index(drop=True)
ranked_display.insert(0, "Rank", ranked_display.index + 1)

# The color scale runs from the lowest displayed macro-F1 (red) to the highest (green).
print("Ranked by mean validation macro-F1. Color compares the displayed scores, from red (lowest) to green (highest).")
display(custom_style(ranked_display).background_gradient(cmap="RdYlGn", subset=["Macro F1"]).set_properties(subset=["Feature pipeline"], **{"white-space": "normal", "max-width": "42em"}))

print(f"Model selection elapsed: {(perf_counter() - comparison_started) / 60:.2f} minutes.")
print("Or ensemble: three balanced subsets × four fresh member models, with four jointly evaluated parameter combinations per compatible feature pipeline.")
print("SVD can produce negative values. Complement Naive Bayes and the ensemble containing it use only the nonnegative feature pipelines.")
ensemble_candidates = ranked_models.loc[ranked_models["Model"].eq("Or balanced occupation ensemble")]
display(ensemble_candidates[["Dataset", "Parameters", "Macro F1", "Accuracy"]].rename(columns={"Dataset": "Or ensemble feature pipeline", "Parameters": "Jointly selected settings"}).reset_index(drop=True))


In [ ]:
# Compare the strongest pipeline and classifier combinations by mean validation F1.
plot_results = ranked_models.head(18).copy()
pipeline_labels = plot_results["Dataset"].str.replace(". ", ".<br>", regex=False)
plot_results["Variant"] = "<b>" + plot_results["Model"] + "</b><br>" + pipeline_labels

# Reuse the scores already measured for the three validation folds.
validation_views = {"Mean across folds": plot_results["Macro F1"].to_numpy()}
for fold in range(len(validation_splits)):
    validation_views[f"Fold {fold + 1}"] = search.cv_results_[f"split{fold}_test_macro_f1"][plot_results.index.to_numpy()]

# Display higher scores in green and lower scores in red.
all_scores = np.concatenate(list(validation_views.values()))
score_colors = [[0, "#c43c36"], [0.5, "#f4bf63"], [1, "#13855d"]]
mean_scores = validation_views["Mean across folds"]
hover_details = plot_results[["Dataset", "Model", "Accuracy", "Fold SD"]].to_numpy()
hover = "Classifier: %{customdata[1]}<br>Feature pipeline: %{customdata[0]}<br>Macro F1: %{x:.3f}<br>Mean accuracy: %{customdata[2]:.3f}<br>F1 fold SD: %{customdata[3]:.3f}<extra></extra>"
figure = go.Figure(go.Bar(x=mean_scores, y=plot_results["Variant"], orientation="h", marker={"color": mean_scores, "colorscale": score_colors, "cmin": float(all_scores.min()), "cmax": float(all_scores.max()), "colorbar": {"title": "Relative F1"}}, texttemplate="%{x:.3f}", textposition="outside", textfont={"size": 15}, cliponaxis=False, customdata=hover_details, hovertemplate=hover))

# Each frame shows an actual fold or the mean, ranked by its own score.
figure.frames = [go.Frame(name=view, data=[go.Bar(x=scores, marker={"color": scores})], layout=go.Layout(yaxis={"categoryorder": "array", "categoryarray": plot_results.iloc[np.argsort(-scores, kind="stable")]["Variant"].tolist(), "autorange": "reversed"})) for view, scores in validation_views.items()]
animation_buttons = [{"label": "Play validation folds", "method": "animate", "args": [list(validation_views)[1:] + ["Mean across folds"], {"frame": {"duration": 1800, "redraw": True}, "transition": {"duration": 500}, "fromcurrent": False}]}, {"label": "Pause", "method": "animate", "args": [[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate"}]}]
animation_steps = [{"label": view, "method": "animate", "args": [[view], {"frame": {"duration": 0, "redraw": True}, "mode": "immediate"}]} for view in validation_views]

# Leave enough room for full pipeline names and provide animation controls.
figure.update_layout(title="Feature pipelines and classifiers: validation macro F1", height=max(1000, 135 * len(plot_results) + 250), margin={"l": 40, "r": 100, "t": 150, "b": 130}, yaxis={"title": None, "automargin": True, "autorange": "reversed", "categoryorder": "array", "categoryarray": plot_results["Variant"].tolist()}, xaxis={"title": "Macro F1", "range": [0, 1.06]}, updatemenus=[{"type": "buttons", "direction": "left", "x": 0, "y": 1.05, "buttons": animation_buttons}], sliders=[{"active": 0, "currentvalue": {"prefix": "Showing: "}, "steps": animation_steps}])
# Explain the ranking and animation before displaying the chart.
race_guide = 'Compare complete feature preparations and classifiers. Higher macro F1 means better balance across occupation codes.\nGreen is a higher score and red a lower score among the results shown. Play shows the three folds, not progress over time.\nPrefer strong results that remain consistent across folds. Hover shows the full preparation.'
plots.add("Validation ranking", figure, race_guide)

# List the selected configuration's fold scores and record counts.
fold_overview = pd.DataFrame([{"Fold": f"Fold {fold + 1}", "Training records": len(fit_rows), "Validation records": len(validation_rows), "Macro F1": search.cv_results_[f"split{fold}_test_macro_f1"][search.best_index_], "Accuracy": search.cv_results_[f"split{fold}_test_accuracy"][search.best_index_]} for fold, (fit_rows, validation_rows) in enumerate(validation_splits)])

# Compare the selected configuration's fold scores visually.
figure = px.bar(fold_overview, x="Fold", y="Macro F1", hover_data=["Accuracy", "Training records", "Validation records"], title="Selected classifier by validation fold")
figure.update_yaxes(range=[0, 1])
plots.add("Selected classifier by validation fold", figure, "Compare the three fold scores. A high mean with a wide spread is less stable across these samples. Folds are different samples, not successive points in time.", details=(fold_overview,))


In [ ]:
# Evaluate the selected configuration once on the held-out records.
selected_pipeline = search.best_estimator_
test_predictions = selected_pipeline.predict(test_matrix)

# Compare with always predicting the most frequent development occupation code.
baseline_predictions = np.repeat(development_targets.mode().iloc[0], len(test_targets))

# Show the recorded and predicted codes for each test record.
held_out_predictions = pd.DataFrame({"ID": record_ids.loc[test_targets.index], "Recorded occupation": test_targets, "Predicted occupation": test_predictions})
held_out_predictions["Correct"] = held_out_predictions["Recorded occupation"].eq(held_out_predictions["Predicted occupation"])

# Calculate exact-code scores and agreement on the first one to four digits.
held_out_metrics = pd.DataFrame([{"Model": name, "Accuracy": accuracy_score(test_targets, predictions), "Macro F1": f1_score(test_targets, predictions, average="macro", zero_division=0)} for name, predictions in [(best_configuration["Classifier"], test_predictions), ("Majority baseline", baseline_predictions)]])
prefix_metrics = pd.DataFrame([{"Digits": width, "Agreement": np.mean([prediction[:width] == actual[:width] for prediction, actual in zip(test_predictions, test_targets)])} for width in range(1, 5)])
display(held_out_metrics)
display(held_out_predictions)

# Compare the same scores and digit-prefix agreements in one compact panel.
figure = make_subplots(rows=1, cols=2, subplot_titles=("Held-out model and majority baseline", "Held-out agreement by occupation-code prefix"))

for metric in ["Accuracy", "Macro F1"]:
    figure.add_trace(go.Bar(x=held_out_metrics["Model"], y=held_out_metrics[metric], name=metric), row=1, col=1)

# The line compares progressively longer occupation-code prefixes.
figure.add_trace(go.Scatter(x=prefix_metrics["Digits"], y=prefix_metrics["Agreement"], mode="lines+markers", line_color="#636efa", showlegend=False), row=1, col=2)

# Keep both vertical axes on the complete 0–1 scale.
figure.update_yaxes(range=[0, 1], tickformat=".0%", title_text="Score", row=1, col=1)
figure.update_yaxes(range=[0, 1], tickformat=".0%", title_text="Agreement", row=1, col=2)
figure.update_xaxes(title_text="Model", row=1, col=1)
figure.update_xaxes(title_text="Leading digits", dtick=1, row=1, col=2)
figure.update_layout(barmode="group", height=420, margin={"t": 80, "b": 75}, legend={"orientation": "h", "y": -0.25})

# Explain the comparison and the meaning of the prefix score.
held_out_guide = 'Check whether the classifier improves occupation prediction on the held-out records.\nLeft: compare accuracy and macro F1 with always predicting the most common code. Taller bars are better.\nRight: agreement with the first one, two, three or all four digits. Four-digit agreement is the complete-code result.'
plots.add("Held-out classification", figure, held_out_guide)


In [ ]:
# Fit the selected configuration on all complete labels.
final_pipeline = clone(selected_pipeline).fit(training_matrix, labelled_targets)
predicted_codes = final_pipeline.predict(prediction_matrix)

if hasattr(final_pipeline, "predict_proba"):
    prediction_scores = final_pipeline.predict_proba(prediction_matrix)
else:
    decision_scores = np.asarray(final_pipeline.decision_function(prediction_matrix))
    prediction_scores = decision_scores if decision_scores.ndim == 2 else np.column_stack([-decision_scores, decision_scores])

# Rank compatible classes using the digits already recorded.
ranked_codes = final_pipeline.classes_[np.argsort(-prediction_scores, axis=1, kind="stable")]
compatible_predictions = [next((candidate for candidate in candidates if all(known == "X" or known == digit for known, digit in zip(recorded, candidate))), pd.NA) for recorded, candidates in zip(partial_targets, ranked_codes)]
prediction_results = pd.DataFrame({"ID": record_ids.loc[partial_targets.index], "Recorded occupation": partial_targets, "Model prediction": predicted_codes, "Digit-compatible prediction": compatible_predictions})

def fit_digit_rules(matrix, codes, feature_names, min_support):
    """
    Find occupation digits shared by every matching record for one feature or a cross-field pair.
    """
    features = sparse.csr_matrix(matrix, copy=True)
    features.eliminate_zeros()
    features.sort_indices()
    fields = [name.split("_", 1)[0] for name in feature_names]
    digits = np.asarray([list(code) for code in codes])
    matching_rows = {}

    for row in range(features.shape[0]):
        present = features.indices[features.indptr[row]:features.indptr[row + 1]]
        for feature in present:
            matching_rows.setdefault((feature,), []).append(row)
        for left, right in combinations(present, 2):
            if fields[left] != fields[right]:
                matching_rows.setdefault((left, right), []).append(row)

    rules = {}
    for terms in sorted(matching_rows, key=len):
        rows = matching_rows[terms]
        if len(rows) < min_support:
            continue

        observed = digits[rows]
        aligned = np.where(np.all(observed == observed[0], axis=0), observed[0], "")
        if len(terms) == 2:
            for feature in terms:
                if (feature,) in rules:
                    aligned[aligned == rules[(feature,)][0]] = ""

        if np.any(aligned != ""):
            rules[terms] = (aligned, len(rows))

    return rules


def predict_digit_rules(matrix, recorded_codes, rules):
    """
    Preserve recorded digits and fill an X when all applicable rules agree on one digit.
    """
    features = sparse.csr_matrix(matrix, copy=True)
    features.eliminate_zeros()
    features.sort_indices()
    predictions = []

    for row, recorded in enumerate(recorded_codes):
        present = features.indices[features.indptr[row]:features.indptr[row + 1]]
        active_rules = [(feature,) for feature in present] + list(combinations(present, 2))
        votes = [rules[terms][0] for terms in active_rules if terms in rules]
        predicted = list(recorded)

        for position, digit in enumerate(recorded):
            supported_digits = {vote[position] for vote in votes if vote[position]}
            if digit == "X" and len(supported_digits) == 1:
                predicted[position] = supported_digits.pop()

        predictions.append("".join(predicted))

    return pd.Series(predictions, index=recorded_codes.index, dtype="string")


minimum_digit_support = 3

# Add the separate industry code only to the statistical digit rules.
industry_codes = data.loc[record_ids.index, "SemelAnafSofi"].astype("string").str.strip().str.replace(r"\.0$", "", regex=True).str.zfill(4)
industry_indicators = pd.get_dummies(industry_codes.where(industry_codes.str.fullmatch(r"[0-9]{4}", na=False)), prefix="SemelAnafSofi", prefix_sep="__", dtype=np.int8, sparse=True)
rule_feature_names = np.r_[feature_names, industry_indicators.columns.to_numpy()]
labelled_rule_matrix = sparse.hstack([training_matrix, industry_indicators.loc[labelled_targets.index].sparse.to_coo()], format="csr")
partial_rule_matrix = sparse.hstack([prediction_matrix, industry_indicators.loc[partial_targets.index].sparse.to_coo()], format="csr")

# Measure rules learned from development records against the same held-out codes.
development_rules = fit_digit_rules(labelled_rule_matrix[development_positions], development_targets, rule_feature_names, minimum_digit_support)
held_out_rule_codes = predict_digit_rules(labelled_rule_matrix[test_positions], pd.Series("XXXX", index=test_targets.index), development_rules)
expected_digits = np.asarray([list(code) for code in test_targets])
proposed_digits = np.asarray([list(code) for code in held_out_rule_codes])
rule_calls = (proposed_digits != "X").sum(axis=0)
rule_hits = (proposed_digits == expected_digits).sum(axis=0)
rule_metrics = pd.DataFrame({"Digit": range(1, 5), "Predicted": rule_calls, "Correct": rule_hits, "Coverage": rule_calls / len(test_targets), "Correct when predicted": np.divide(rule_hits, rule_calls, out=np.full(4, np.nan), where=rule_calls > 0)})

# Refit the rules on complete codes before applying them to the incomplete records.
final_rules = fit_digit_rules(labelled_rule_matrix, labelled_targets, rule_feature_names, minimum_digit_support)
prediction_results["Industry code"] = industry_codes
prediction_results["Rule-based occupation"] = predict_digit_rules(partial_rule_matrix, partial_targets, final_rules)
print("Rule-based occupation fills a missing digit when applicable rules agree. X means unresolved.")
display(prediction_results)


In [ ]:
# Connect model selection, held-out performance and the evidence behind digit rules.
selected_color = "#0f766e"
baseline_color = "#94a3b8"
rule_color = "#c2410c"
digit_heat_colors = ["#ffedd5", "#fb923c", "#b91c1c"]

# Select the four strongest validation configurations.
shortlist = ranked_models.head(4)
model_labels = shortlist["Dataset"].str.replace(". ", ".<br>", regex=False) + "<br>" + shortlist["Model"]
selected = best_configuration["Classifier"]
panel_titles = ("Model selection · grouped CV macro-F1 ± fold SD", f"Held-out model vs baseline · n={len(test_targets)}", "Held-out occupation prefix agreement", "Held-out digit rules · coverage and correctness", "100% alignment in development records · strongest rules")
figure = make_subplots(rows=3, cols=2, specs=[[{}, {}], [{}, {}], [{"colspan": 2}, None]], subplot_titles=panel_titles, row_heights=[0.38, 0.22, 0.40], vertical_spacing=0.09, horizontal_spacing=0.18)
# Show their mean macro F1 and fold variation.
figure.add_trace(go.Bar(x=shortlist["Macro F1"], y=model_labels, orientation="h", marker_color=np.where(shortlist.index == search.best_index_, selected_color, baseline_color), error_x={"array": shortlist["Fold SD"]}, texttemplate="%{x:.2f}", textposition="outside", showlegend=False), row=1, col=1)

# Compare the selected model with the most-common-code reference on the holdout.
for color, row in zip([selected_color, baseline_color], held_out_metrics.itertuples(index=False, name=None)):
    model_name, accuracy, macro_f1 = row
    figure.add_trace(go.Bar(x=["Macro F1", "Accuracy"], y=[macro_f1, accuracy], name=model_name, marker_color=color, texttemplate="%{y:.0%}", textposition="outside"), row=1, col=2)

# Compare prefixes of increasing length against recorded occupation codes.
figure.add_trace(go.Scatter(x=prefix_metrics["Digits"], y=prefix_metrics["Agreement"], mode="lines+markers+text", texttemplate="%{y:.0%}", textposition="top center", line_color=selected_color, showlegend=False), row=2, col=1)

# Show both how often digit rules predict and how often their predictions are correct.
for metric, color in [("Coverage", baseline_color), ("Correct when predicted", rule_color)]:
    figure.add_trace(go.Bar(x=rule_metrics["Digit"], y=rule_metrics[metric], name=metric, marker_color=color, customdata=rule_metrics[["Predicted", "Correct"]], hovertemplate="%{y:.0%}<br>Predicted: %{customdata[0]}<br>Correct: %{customdata[1]}"), row=2, col=2)

# Keep single-field evidence, cross-field combinations and the industry link visible.
ordered_rules = sorted(development_rules, key=lambda terms: -development_rules[terms][1])
industry_rules = [terms for terms in ordered_rules if any(rule_feature_names[term].startswith("SemelAnafSofi__") for term in terms)]
single_rules = [terms for terms in ordered_rules if len(terms) == 1]
paired_rules = [terms for terms in ordered_rules if len(terms) == 2]
visible_rules = list(dict.fromkeys(industry_rules[:4] + single_rules[:4] + paired_rules[:6]))
heat_digits = np.asarray([development_rules[terms][0] for terms in visible_rules]).reshape(-1, 4)
heat_counts = np.asarray([development_rules[terms][1] for terms in visible_rules])
heat_labels = [" + ".join(rule_feature_names[list(terms)]) for terms in visible_rules]
heat_rows = [f"{label} · n={support}" for label, support in zip(heat_labels, heat_counts)]
figure.add_trace(go.Heatmap(x=[f"Digit {digit}" for digit in range(1, 5)], y=heat_rows, z=np.where(heat_digits != "", heat_counts[:, None], np.nan), text=heat_digits, texttemplate="%{text}", textfont={"size": 16, "color": "#431407"}, colorscale=digit_heat_colors, colorbar={"title": "Records", "len": 0.35, "y": 0.18}, xgap=3, ygap=3, hoverongaps=False, hovertemplate="%{y}<br>%{x} = %{text}<br>All %{z} matching records agree<extra></extra>"), row=3, col=1)

# Use extra space and larger text for the pipeline names and rules.
figure.update_layout(title=f"Occupation classification results · {selected}", height=1900, barmode="group", legend={"orientation": "h", "y": -0.06, "font": {"size": 15}}, margin={"t": 130, "b": 110, "r": 90})
figure.update_xaxes(range=[0, 1.05], title_text="Mean macro-F1", row=1, col=1)
figure.update_xaxes(dtick=1, title_text="Leading digits", row=2, col=1)
figure.update_xaxes(dtick=1, title_text="Occupation-code digit", row=2, col=2)
figure.update_yaxes(range=[0, 1.15], tickformat=".0%", col=2)
figure.update_yaxes(range=[0, 1.12], tickformat=".0%", row=2, col=1)
figure.update_yaxes(autorange="reversed", automargin=True, row=1, col=1)
figure.update_yaxes(autorange="reversed", automargin=True, row=3, col=1)
# Explain the values currently displayed and the comparisons they support.
selected_holdout = held_out_metrics.iloc[0]
reference_holdout = held_out_metrics.iloc[1]
prefix_one = prefix_metrics.loc[prefix_metrics["Digits"].eq(1), "Agreement"].iloc[0]
prefix_four = prefix_metrics.loc[prefix_metrics["Digits"].eq(4), "Agreement"].iloc[0]

# Report current results alongside examples of other possible patterns.

summary_guide = f"Compare model selection, held-out classification, code prefixes and digit rules.\nThe selected model scores {selected_holdout['Accuracy']:.0%} accuracy versus {reference_holdout['Accuracy']:.0%} for the majority baseline. First-digit agreement is {prefix_one:.0%}, complete-code agreement {prefix_four:.0%}.\nHigher scores are better. Development rules that match perfectly can still fail on new records. Feature preparation preceded the split, so this comparison is exploratory."
plots.add("Classification summary", figure, summary_guide)


In [ ]:
# Cluster each original text field independently before combining its one-hot features.
class ClusterMembershipFeatures(TransformerMixin, BaseEstimator):
    """
    Replace each field's TF-IDF vector with its independently selected cluster label.

    feature_names identifies original text fields by the prefix before two
    underscores. For every field, centroids and the cluster count use only the
    supplied fitting rows. Calinski-Harabasz selects among 2 to 10 Euclidean
    k-means clusters. L2 normalization is optional and also applied field by
    field. The output concatenates the independently one-hot encoded labels
    with the unchanged categorical and numeric survey features.
    Missing text produces zeros for that field. A field with insufficient
    distinct fitting vectors uses one observed cluster and reports no CH score.
    """

    def __init__(self, feature_names=None, feature_positions=None, normalize=True, max_clusters=10, random_state=42):
        self.feature_names = feature_names
        self.feature_positions = feature_positions
        self.normalize = normalize
        self.max_clusters = max_clusters
        self.random_state = random_state

    def fit(self, X, y=None):
        self.__dict__.pop("field_models_", None)
        features = sparse.csr_matrix(X)
        names = np.asarray(self.feature_names, dtype=str)
        positions = self.feature_positions if self.feature_positions is not None else np.flatnonzero(np.char.find(names, "__") >= 0)
        field_names = list(dict.fromkeys(names[position].split("__", 1)[0] for position in positions))
        self.survey_feature_positions_ = np.setdiff1d(np.arange(features.shape[1]), positions)
        field_models = []
        selection_rows = []
        field_rows = []

        for field in field_names:
            field_positions = np.asarray([position for position in positions if names[position].startswith(field + "__")])
            field_features = features[:, field_positions].copy()
            field_features.eliminate_zeros()
            field_features.sort_indices()
            normalizer = Normalizer() if self.normalize else None
            normalized = normalizer.fit_transform(field_features) if self.normalize else field_features
            nonempty = np.flatnonzero(normalized.getnnz(axis=1))
            fitting = normalized[nonempty]
            row_keys = {(tuple(fitting.indices[start:stop]), tuple(fitting.data[start:stop])) for start, stop in zip(fitting.indptr[:-1], fitting.indptr[1:])}
            maximum = min(self.max_clusters, len(row_keys), fitting.shape[0] - 1)
            best_model = None
            best_score = -np.inf
            field_selection = []

            if maximum >= 2:
                # Only the small field matrix is made dense for sklearn's CH scorer.
                # K-means itself receives the sparse CSR vectors.
                scoring_vectors = fitting.toarray()
                for clusters in range(2, maximum + 1):
                    candidate = KMeans(n_clusters=clusters, n_init=10, algorithm="lloyd", random_state=self.random_state).fit(fitting)
                    if len(np.unique(candidate.labels_)) != clusters:
                        continue
                    score = calinski_harabasz_score(scoring_vectors, candidate.labels_)
                    field_selection.append({"Field": field, "Clusters": clusters, "Calinski-Harabasz": score})
                    if score > best_score:
                        best_model, best_score = candidate, score

            clusters = best_model.n_clusters if best_model is not None else 1
            encoder = OneHotEncoder(categories=[np.arange(clusters)], handle_unknown="ignore")
            fitted_labels = best_model.labels_ if best_model is not None else np.zeros(max(1, len(nonempty)), dtype=int)
            encoder.fit(fitted_labels.reshape(-1, 1))
            field_models.append({"field": field, "positions": field_positions, "normalizer": normalizer, "model": best_model, "encoder": encoder, "clusters": clusters, "has_fitting_text": len(nonempty) > 0})
            if not field_selection:
                field_selection.append({"Field": field, "Clusters": 1, "Calinski-Harabasz": np.nan})
            for row in field_selection:
                row["Selected"] = row["Clusters"] == clusters
                selection_rows.append(row)
            field_rows.append({"Field": field, "TF-IDF features": len(field_positions), "Nonempty fitting records": len(nonempty), "Distinct fitting vectors": len(row_keys), "Selected clusters": clusters, "Calinski-Harabasz": best_score if best_model is not None else np.nan})

        self.field_models_ = field_models
        self.field_summary_ = pd.DataFrame(field_rows)
        self.selection_ = pd.DataFrame(selection_rows)
        self.n_clusters_ = sum(field["clusters"] for field in field_models)
        return self

    def transform(self, X):
        check_is_fitted(self, "field_models_")
        features = sparse.csr_matrix(X)
        encoded_fields = []
        for field in self.field_models_:
            field_features = features[:, field["positions"]].copy()
            field_features.eliminate_zeros()
            field_features.sort_indices()
            normalized = field["normalizer"].transform(field_features) if field["normalizer"] is not None else field_features
            nonempty = np.flatnonzero(normalized.getnnz(axis=1))
            if len(nonempty) == 0 or not field["has_fitting_text"]:
                encoded_fields.append(sparse.csr_matrix((features.shape[0], field["clusters"])))
                continue
            labels = field["model"].predict(normalized[nonempty]) if field["model"] is not None else np.zeros(len(nonempty), dtype=int)
            encoded = field["encoder"].transform(labels.reshape(-1, 1)).tocoo()
            encoded_fields.append(sparse.csr_matrix((encoded.data, (nonempty[encoded.row], encoded.col)), shape=(features.shape[0], field["clusters"])))
        # Replace only TF-IDF columns and preserve the remaining survey inputs.
        return sparse.hstack(encoded_fields + [features[:, self.survey_feature_positions_]], format="csr")


cluster_paths = {
    "For seven text fields: clean the text, use LaBSE matching on four fields, then apply TF-IDF. Then cluster each text field with k-means, selecting 2–10 clusters by Calinski-Harabasz. Then one-hot encode each cluster assignment. Separately: one-hot encode categorical fields, including workplace settlement, and prepare numeric fields. Combine the results": False,
    "For seven text fields: clean the text, use LaBSE matching on four fields, then apply TF-IDF. Then L2-normalize each text field. Then cluster each text field with k-means, selecting 2–10 clusters by Calinski-Harabasz. Then one-hot encode each cluster assignment. Separately: one-hot encode categorical fields, including workplace settlement, and prepare numeric fields. Combine the results": True,
}
text_feature_positions = np.flatnonzero(np.char.find(feature_names.astype(str), "__") >= 0)
# Keep the same four individual classifiers and Or's ensemble in both representation comparisons.
cluster_model_families = model_families
cluster_comparison_started = perf_counter()
cluster_validation_inputs = {}
cluster_partition_rows = []
cluster_candidate_rows = []
cluster_fold_rows = []
cluster_candidate_models = {}
cluster_diagnostic_folds = {}

# Learn each partition's centroids once and share its cluster features across classifiers.
for preparation, normalize in cluster_paths.items():
    validation_inputs = []
    for fold, (fit_rows, validation_rows) in enumerate(validation_splits, start=1):
        encoder = ClusterMembershipFeatures(feature_names=feature_names, feature_positions=text_feature_positions, normalize=normalize, random_state=SEED).fit(development_matrix[fit_rows])
        validation_inputs.append((encoder.transform(development_matrix[fit_rows]), encoder.transform(development_matrix[validation_rows])))
        cluster_partition_rows.extend(encoder.field_summary_.assign(**{"Feature preparation": preparation, "Partition": f"Validation round {fold}", "Fitting records": len(fit_rows)}).to_dict("records"))
    cluster_validation_inputs[preparation] = validation_inputs

    # Fit each parameter choice once per validation round on cluster indicators and the separately prepared survey features.
    # Keep its fitted model and predictions available for ROC and learning-curve diagnostics.
    for family, (classifier, parameter_grid) in cluster_model_families.items():
        for parameters in ParameterGrid(parameter_grid):
            model_parameters = {name.removeprefix("classifier__"): value for name, value in parameters.items()}
            fold_scores = []
            fitted_models = []
            for fold, ((fit_rows, validation_rows), (fitting_features, validation_features)) in enumerate(zip(validation_splits, validation_inputs), start=1):
                fitted = clone(classifier).set_params(**model_parameters).fit(fitting_features, development_targets.iloc[fit_rows])
                predictions = fitted.predict(validation_features)
                score = {"Feature preparation": preparation, "Classifier": family, "Parameters": json.dumps(parameters), "Fold": fold, "Macro F1": f1_score(development_targets.iloc[validation_rows], predictions, average="macro", zero_division=0), "Accuracy": accuracy_score(development_targets.iloc[validation_rows], predictions)}
                fold_scores.append(score)
                cluster_fold_rows.append(score)
                fitted_models.append({"estimator": fitted, "features": validation_features, "expected": development_targets.iloc[validation_rows], "fitting_targets": development_targets.iloc[fit_rows], "predictions": predictions, "fold": fold, "macro_f1": score["Macro F1"], "accuracy": score["Accuracy"]})
            cluster_candidate_rows.append({"Feature preparation": preparation, "Classifier": family, "Parameters": json.dumps(parameters), "Macro F1": np.mean([row["Macro F1"] for row in fold_scores]), "Fold SD": np.std([row["Macro F1"] for row in fold_scores]), "Accuracy": np.mean([row["Accuracy"] for row in fold_scores])})
            cluster_candidate_models[(preparation, family, json.dumps(parameters))] = fitted_models

cluster_candidates = pd.DataFrame(cluster_candidate_rows)
cluster_validation_results = cluster_candidates.sort_values("Macro F1", ascending=False, kind="stable").drop_duplicates(["Feature preparation", "Classifier"])
cluster_fold_results = pd.DataFrame(cluster_fold_rows)
for result in cluster_validation_results.to_dict("records"):
    key = (result["Feature preparation"], result["Classifier"])
    cluster_diagnostic_folds[key] = cluster_candidate_models[(*key, result["Parameters"])]
print("Each original text field has its own cluster model. A record activates one cluster column for each nonempty field.")
print("Only TF-IDF columns are replaced. Existing categorical and numeric survey features remain in both cluster variants.")
print("For each field, sklearn Calinski-Harabasz selects the best valid cluster count from 2 to 10 using fitting records only.")
display(cluster_validation_results[["Feature preparation", "Classifier", "Macro F1", "Accuracy"]].reset_index(drop=True))

# Reuse one development clustering for every classifier on the same fixed test records.
cluster_test_rows = []
cluster_development_encoders = {}
for preparation, normalize in cluster_paths.items():
    encoder = ClusterMembershipFeatures(feature_names=feature_names, feature_positions=text_feature_positions, normalize=normalize, random_state=SEED).fit(development_matrix)
    cluster_development_encoders[preparation] = encoder
    fitting_features = encoder.transform(development_matrix)
    test_features = encoder.transform(test_matrix)
    cluster_partition_rows.extend(encoder.field_summary_.assign(**{"Feature preparation": preparation, "Partition": "Development for test", "Fitting records": len(development_targets)}).to_dict("records"))
    for result in cluster_validation_results.loc[cluster_validation_results["Feature preparation"].eq(preparation)].to_dict("records"):
        classifier = cluster_model_families[result["Classifier"]][0]
        parameters = {name.removeprefix("classifier__"): value for name, value in json.loads(result["Parameters"]).items()}
        fitted = clone(classifier).set_params(**parameters).fit(fitting_features, development_targets)
        predictions = fitted.predict(test_features)
        cluster_test_rows.append({"Feature preparation": preparation, "Classifier": result["Classifier"], "Correct": int(np.sum(predictions == test_targets.to_numpy())), "Test records": len(test_targets), "Accuracy": accuracy_score(test_targets, predictions), "Macro F1": f1_score(test_targets, predictions, average="macro", zero_division=0)})
cluster_test_results = pd.DataFrame(cluster_test_rows)
display(cluster_test_results)
display(pd.DataFrame(cluster_partition_rows))
display(pd.DataFrame([{"Feature pipeline": preparation, "Original TF-IDF columns": len(text_feature_positions), "Cluster one-hot columns": encoder.n_clusters_, "Other survey columns": len(encoder.survey_feature_positions_), "Total model columns": encoder.n_clusters_ + len(encoder.survey_feature_positions_)} for preparation, encoder in cluster_development_encoders.items()]))

# Refit the validation-selected path and classifier on complete codes for prediction.
cluster_best = cluster_validation_results.iloc[0]
cluster_final_encoder = ClusterMembershipFeatures(feature_names=feature_names, feature_positions=text_feature_positions, normalize=cluster_paths[cluster_best["Feature preparation"]], random_state=SEED).fit(training_matrix)
cluster_final_parameters = {name.removeprefix("classifier__"): value for name, value in json.loads(cluster_best["Parameters"]).items()}
cluster_final_classifier = clone(cluster_model_families[cluster_best["Classifier"]][0]).set_params(**cluster_final_parameters).fit(cluster_final_encoder.transform(training_matrix), labelled_targets)
cluster_classifier_pipeline = Pipeline([("clusters", cluster_final_encoder), ("classifier", cluster_final_classifier)])
cluster_predictions = pd.DataFrame({"ID": record_ids.loc[partial_targets.index], "Recorded occupation": partial_targets, "Cluster-compressed prediction": cluster_classifier_pipeline.predict(prediction_matrix)})
display(cluster_predictions)

# Compare the candidate cluster counts within each original text field.
cluster_selection_curves = pd.concat([encoder.selection_.assign(**{"Feature preparation": preparation, "Clustering space": "L2-normalized TF-IDF vectors" if encoder.normalize else "TF-IDF vectors"}) for preparation, encoder in cluster_development_encoders.items()], ignore_index=True)

# Higher scores indicate better cluster separation relative to internal spread.
figure = px.line(cluster_selection_curves, x="Clusters", y="Calinski-Harabasz", color="Field", facet_col="Clustering space", markers=True, title="Training-only Calinski-Harabasz selection for each text field")

# Make the tested integer cluster counts, field legend, and axis labels readable.
figure.update_xaxes(dtick=1)
figure.update_layout(height=650, legend={"orientation": "h", "y": -0.22}, margin={"b": 130})

# Show how many clusters were selected for each original text field.
selected_cluster_counts = pd.concat([encoder.field_summary_.set_index("Field")["Selected clusters"].rename("L2 normalized per field" if encoder.normalize else "Without L2 normalization") for encoder in cluster_development_encoders.values()], axis=1)

# Compare each preparation's best classifier on the same validation rounds.
base_preparation = dataset_options["Dataset"].iloc[0]
base_scores = ranked_models.loc[ranked_models["Dataset"].eq(base_preparation)].rename(columns={"Dataset": "Feature preparation", "Model": "Classifier"})
cluster_comparison_scores = pd.concat([base_scores, cluster_validation_results], ignore_index=True)
cluster_comparison_scores = cluster_comparison_scores.sort_values("Macro F1", ascending=False, kind="stable").drop_duplicates("Feature preparation")
cluster_comparison_summary = []

for row in cluster_comparison_scores.to_dict("records"):
    preparation = row["Feature preparation"]
    classifier = row["Classifier"]
    if preparation == base_preparation:
        preparation_name = "Text TF-IDF without clustering; categorical and numeric fields prepared separately and combined"
        test_score = held_out_metrics.loc[held_out_metrics["Model"].eq(classifier)] if best_configuration["Dataset"] == base_preparation else held_out_metrics.iloc[0:0]
        test_correct = int(np.sum(test_predictions == test_targets.to_numpy())) if not test_score.empty else None
    else:
        preparation_name = "Text TF-IDF, then per-field k-means and one-hot cluster assignments. Categorical and numeric fields prepared separately and combined"
        if cluster_paths[preparation]:
            preparation_name = "Text TF-IDF, then per-field L2 normalization, k-means and one-hot cluster assignments. Categorical and numeric fields prepared separately and combined"
        test_score = cluster_test_results.loc[cluster_test_results["Feature preparation"].eq(preparation) & cluster_test_results["Classifier"].eq(classifier)]
        test_correct = int(test_score["Correct"].iloc[0]) if not test_score.empty else None
    cluster_comparison_summary.append({"Feature preparation": preparation_name, "Best classifier": classifier, "Validation macro F1": row["Macro F1"], "Validation accuracy": row["Accuracy"], "Test macro F1": test_score["Macro F1"].iloc[0] if not test_score.empty else np.nan, "Test accuracy": test_score["Accuracy"].iloc[0] if not test_score.empty else np.nan, "Test correct": f"{test_correct}/{len(test_targets)}" if test_correct is not None else "Not evaluated"})

# Keep validation and held-out test results distinct; missing scores remain unavailable.
cluster_comparison_summary = pd.DataFrame(cluster_comparison_summary)
cluster_guide = 'Choose a cluster count for each text field using its highest Calinski–Harabasz score.\nEach line is one field. A clear peak favors that tested count. A flat line gives little preference. A peak at 10 leaves larger counts untested.\nSeparation is not prediction accuracy. The tables show whether the selected clusters help classify occupation codes.'
plots.add("Cluster separation and classification", figure, cluster_guide, details=(selected_cluster_counts, custom_style(cluster_comparison_summary).set_properties(subset=["Feature preparation"], **{"white-space": "normal", "max-width": "38em"})))

# Report the time spent on clustering and classifier comparisons.
print(f"Field clustering and classifier comparison elapsed: {(perf_counter() - cluster_comparison_started) / 60:.2f} minutes.")


In [ ]:
# Compare cluster counts by their downstream classification results.
def fit_clusters_at_count(fitting_matrix, validation_matrix, feature_names, text_positions, requested_count, normalize):
    """Fit one k-means model per text field, then keep the remaining survey features."""
    names = np.asarray(feature_names, dtype=str)
    fields = list(dict.fromkeys(names[position].split("__", 1)[0] for position in text_positions))
    other_positions = np.setdiff1d(np.arange(len(names)), text_positions)
    fitting_parts = []
    validation_parts = []
    selected_counts = {}

    for field in fields:
        # Keep the vocabulary of each text field separate.
        positions = np.asarray([position for position in text_positions if names[position].startswith(field + "__")])
        fitting = fitting_matrix[:, positions].tocsr(copy=True)
        validation = validation_matrix[:, positions].tocsr(copy=True)
        fitting.eliminate_zeros()
        validation.eliminate_zeros()

        # The normalized path scales each nonempty text vector, not the survey fields.
        if normalize:
            normalizer = Normalizer()
            fitting = normalizer.fit_transform(fitting)
            validation = normalizer.transform(validation)

        fitting_rows = np.flatnonzero(fitting.getnnz(axis=1))
        validation_rows = np.flatnonzero(validation.getnnz(axis=1))
        fitting_values = fitting[fitting_rows]
        distinct = len(np.unique(fitting_values.toarray(), axis=0)) if len(fitting_rows) else 0
        count = min(requested_count, distinct)
        selected_counts[field] = count

        # A missing field stays all-zero. A single distinct value forms one cluster.
        if count == 0:
            fitting_parts.append(sparse.csr_matrix((fitting.shape[0], 1)))
            validation_parts.append(sparse.csr_matrix((validation.shape[0], 1)))
            continue

        if count == 1:
            fitting_labels = np.zeros(len(fitting_rows), dtype=int)
            validation_labels = np.zeros(len(validation_rows), dtype=int)
        else:
            cluster_model = KMeans(n_clusters=count, n_init=10, algorithm="lloyd", random_state=SEED).fit(fitting_values)
            fitting_labels = cluster_model.labels_
            validation_labels = cluster_model.predict(validation[validation_rows]) if len(validation_rows) else np.array([], dtype=int)

        # Report clusters actually represented among fitting records.
        selected_counts[field] = len(np.unique(fitting_labels))

        # Each nonempty answer activates exactly one cluster indicator.
        fitting_parts.append(sparse.csr_matrix((np.ones(len(fitting_rows)), (fitting_rows, fitting_labels)), shape=(fitting.shape[0], count)))
        validation_parts.append(sparse.csr_matrix((np.ones(len(validation_rows)), (validation_rows, validation_labels)), shape=(validation.shape[0], count)))

    # Keep the same one-hot categorical and numeric survey features as before.
    fitting_parts.append(fitting_matrix[:, other_positions])
    validation_parts.append(validation_matrix[:, other_positions])
    return sparse.hstack(fitting_parts, format="csr"), sparse.hstack(validation_parts, format="csr"), selected_counts


# Keep one previously selected classifier and its parameters per clustering path.
cluster_count_classifiers = cluster_validation_results.sort_values("Macro F1", ascending=False, kind="stable").drop_duplicates("Feature preparation")
cluster_count_folds = []
cluster_count_started = perf_counter()

for configuration in cluster_count_classifiers.to_dict("records"):
    preparation = configuration["Feature preparation"]
    classifier_name = configuration["Classifier"]
    parameters = {name.removeprefix("classifier__"): value for name, value in json.loads(configuration["Parameters"]).items()}
    classifier = cluster_model_families[classifier_name][0]

    # Each candidate uses the same grouped validation rounds as the previous comparison.
    for requested_count in range(2, 11):
        for fold, (fit_rows, validation_rows) in enumerate(validation_splits, start=1):
            fitting_features, validation_features, field_counts = fit_clusters_at_count(development_matrix[fit_rows], development_matrix[validation_rows], feature_names, text_feature_positions, requested_count, cluster_paths[preparation])
            fitted = clone(classifier).set_params(**parameters).fit(fitting_features, development_targets.iloc[fit_rows])
            expected = development_targets.iloc[validation_rows]
            predictions = fitted.predict(validation_features)
            cluster_count_folds.append({"Feature preparation": preparation, "Classifier": classifier_name, "Requested clusters": requested_count, "Fold": fold, "Macro F1": f1_score(expected, predictions, average="macro", zero_division=0), "Accuracy": accuracy_score(expected, predictions), "Field cluster counts": json.dumps(field_counts, ensure_ascii=False)})

# Average the three scores for each requested count and each feature preparation.
cluster_count_fold_results = pd.DataFrame(cluster_count_folds)
cluster_count_results = cluster_count_fold_results.groupby(["Feature preparation", "Classifier", "Requested clusters"], sort=False).agg(**{"Macro F1": ("Macro F1", "mean"), "Fold SD": ("Macro F1", lambda scores: scores.std(ddof=0)), "Accuracy": ("Accuracy", "mean"), "Accuracy SD": ("Accuracy", lambda scores: scores.std(ddof=0))}).reset_index()
cluster_count_baseline = cluster_count_classifiers[["Feature preparation", "Classifier", "Macro F1", "Accuracy"]].copy()
clustering_space_labels = {preparation: "TF-IDF with L2 per field" if normalize else "TF-IDF without L2" for preparation, normalize in cluster_paths.items()}
cluster_count_results["Text vector preparation"] = cluster_count_results["Feature preparation"].map(clustering_space_labels)


# Plot both classification scores against the requested count, next to the CH baseline.
cluster_count_colors = {"TF-IDF without L2": "#b96e30", "TF-IDF with L2 per field": "#177b68"}
cluster_count_figure = make_subplots(rows=1, cols=2, subplot_titles=("Validation macro F1", "Validation accuracy"))
for column, (metric, deviation) in enumerate([("Macro F1", "Fold SD"), ("Accuracy", "Accuracy SD")], start=1):
    for preparation in cluster_paths:
        label = clustering_space_labels[preparation]
        results = cluster_count_results.loc[cluster_count_results["Feature preparation"].eq(preparation)].sort_values("Requested clusters")
        baseline = cluster_count_baseline.loc[cluster_count_baseline["Feature preparation"].eq(preparation)].iloc[0]
        classifier_name = baseline["Classifier"]
        cluster_count_figure.add_trace(go.Scatter(x=results["Requested clusters"], y=results[metric], mode="lines+markers", name=f"{label} · {classifier_name}", legendgroup=label, showlegend=column == 1, line={"color": cluster_count_colors[label]}, error_y={"type": "data", "array": results[deviation]}, hovertemplate=f"{label}<br>Requested clusters: %{{x}}<br>{metric}: %{{y:.3f}}<extra>{classifier_name}</extra>"), row=1, col=column)
        cluster_count_figure.add_trace(go.Scatter(x=[2, 10], y=[baseline[metric], baseline[metric]], mode="lines", name=f"CH-selected per field · {label}", legendgroup=f"CH {label}", showlegend=column == 1, line={"color": cluster_count_colors[label], "dash": "dash"}, hovertemplate=f"CH-selected per field<br>{classifier_name}<br>{metric}: {baseline[metric]:.3f}<extra></extra>"), row=1, col=column)
cluster_count_figure.update_xaxes(dtick=1, title="Requested clusters per text field")
cluster_count_figure.update_yaxes(range=[0, 1], tickformat=".0%")
cluster_count_figure.update_layout(title="Classification performance by requested clusters per text field", height=650, legend={"orientation": "h", "y": -0.25}, margin={"t": 110, "b": 170})

# Select the requested count from validation only, then evaluate it on the separate test records.
cluster_count_best = cluster_count_results.sort_values("Macro F1", ascending=False, kind="stable").drop_duplicates("Feature preparation")
cluster_count_test_rows = []
cluster_count_test_predictions = {}

# Show the best tested count and its validation score for each path.

for selection in cluster_count_best.to_dict("records"):
    preparation = selection["Feature preparation"]
    classifier_name = selection["Classifier"]
    requested_count = selection["Requested clusters"]
    configuration = cluster_count_classifiers.loc[cluster_count_classifiers["Feature preparation"].eq(preparation)].iloc[0]
    parameters = {name.removeprefix("classifier__"): value for name, value in json.loads(configuration["Parameters"]).items()}
    classifier = cluster_model_families[classifier_name][0]

    # Fit clusters on development records only before scoring the held-out records.
    fitting_features, test_features, field_counts = fit_clusters_at_count(development_matrix, test_matrix, feature_names, text_feature_positions, requested_count, cluster_paths[preparation])
    fitted = clone(classifier).set_params(**parameters).fit(fitting_features, development_targets)
    cluster_count_test_prediction = fitted.predict(test_features)
    cluster_count_test_rows.append({"Feature preparation": preparation, "Classifier": classifier_name, "Requested clusters": requested_count, "Test records": len(test_targets), "Correct": int(np.sum(cluster_count_test_prediction == test_targets.to_numpy())), "Accuracy": accuracy_score(test_targets, cluster_count_test_prediction), "Macro F1": f1_score(test_targets, cluster_count_test_prediction, average="macro", zero_division=0), "Clusters per field": json.dumps(field_counts, ensure_ascii=False)})
    cluster_count_test_predictions[preparation] = pd.DataFrame({"ID": record_ids.loc[test_targets.index], "Recorded occupation": test_targets, "Predicted occupation": cluster_count_test_prediction})

cluster_count_test_results = pd.DataFrame(cluster_count_test_rows)
cluster_count_test_baseline = cluster_test_results.merge(cluster_count_classifiers[["Feature preparation", "Classifier"]], on=["Feature preparation", "Classifier"])

print(f"Cluster-count classification comparison elapsed: {(perf_counter() - cluster_count_started) / 60:.2f} minutes.")

# Register the completed comparison once all test results are available.
count_guide = 'Test whether a different cluster count improves occupation classification.\nHigher macro F1 and accuracy are better. Dashed lines show the per-field Calinski–Harabasz baseline. Error bars show variation across folds.\nThe best validation point must also be checked against the held-out results below. These reused selection folds and previously examined test records make the comparison exploratory.'
plots.add("Cluster count and classification", cluster_count_figure, count_guide, details=(cluster_count_best[["Text vector preparation", "Classifier", "Requested clusters", "Macro F1", "Accuracy"]].reset_index(drop=True), cluster_count_test_results, cluster_count_test_baseline))


In [ ]:
# Combine text-field clusters, one-hot survey categories, and fixed age bands.
# Age is represented by decade indicators, not by treating its numeric value as a binary indicator.
survey_indicator_prefixes = ("YeshuvAvoda_", "MaamadAvoda_", "MakorSachar_", "TeudaGvoha_")
survey_indicator_positions = np.asarray([position for position, name in enumerate(feature_names.astype(str)) if name.startswith(survey_indicator_prefixes)])
management_position = np.flatnonzero(feature_names.astype(str) == "MenahelEtMi")[0]
age_position = np.flatnonzero(feature_names.astype(str) == "Gil")[0]
age_band_edges = np.r_[18, np.arange(20, 121, 10), 121]


def cluster_digit_features(encoder, matrix):
    """Combine cluster membership, one-hot survey categories, and age bands."""
    clustered = encoder.transform(matrix)[:, :encoder.n_clusters_]
    categories = matrix[:, survey_indicator_positions]
    cluster_names = [f"{field['field']}__cluster_{cluster}" for field in encoder.field_models_ for cluster in range(field["clusters"])]
    category_names = feature_names[survey_indicator_positions].astype(str).tolist()
    # Keep each management code distinct. Upstream preparation maps missing values to 0.
    management_codes = matrix[:, management_position].toarray()
    management = sparse.csr_matrix((management_codes == np.arange(4)).astype(np.int8))
    management_names = [f"MenahelEtMi_{code}" for code in range(4)]
    # Fixed decade bands use the age already prepared by the notebook.
    ages = matrix[:, age_position].toarray()
    age_bands = sparse.csr_matrix(((ages >= age_band_edges[:-1]) & (ages < age_band_edges[1:])).astype(np.int8))
    age_names = [f"Gil_{start}" if end == start + 1 else f"Gil_{start}–{end - 1}" for start, end in zip(age_band_edges[:-1], age_band_edges[1:])]
    names = cluster_names + category_names + management_names + age_names
    fields = [name.split("_", 1)[0] for name in names]
    return sparse.hstack([clustered, categories, management, age_bands], format="csr"), names, fields


def fit_cluster_digit_rules(features, targets, fields, min_support=3):
    """Find one-, two-, and three-field combinations sharing an occupation digit."""
    features = sparse.csr_matrix(features, copy=True)
    features.eliminate_zeros()
    features.sort_indices()
    target_digits = np.asarray([list(code) for code in targets])
    matching_rows = {}

    # Count records for each combination of indicators from distinct fields.
    for row in range(features.shape[0]):
        active = features.indices[features.indptr[row]:features.indptr[row + 1]]
        for size in (1, 2, 3):
            for terms in combinations(active, size):
                if len({fields[term] for term in terms}) == size:
                    matching_rows.setdefault(terms, []).append(row)

    rules = {}
    for terms, rows in matching_rows.items():
        if len(rows) < min_support:
            continue
        observed = target_digits[rows]
        aligned_digits = np.where(np.all(observed == observed[0], axis=0), observed[0], "")
        if np.any(aligned_digits != ""):
            rules[terms] = (aligned_digits, len(rows))
    return rules


def transform_cluster_digit_indicators(features, rules, fields, index):
    """Unite agreeing rules into one binary indicator per digit position and value."""
    features = sparse.csr_matrix(features, copy=True)
    features.eliminate_zeros()
    features.sort_indices()
    indicator_pairs = sorted({(position, digit) for digits, _ in rules.values() for position, digit in enumerate(digits) if digit})
    indicator_positions = {pair: number for number, pair in enumerate(indicator_pairs)}
    indicator_values = np.zeros((features.shape[0], len(indicator_pairs)), dtype=np.int8)

    # The logical OR joins every matching rule that supports the same digit.
    for row in range(features.shape[0]):
        active = features.indices[features.indptr[row]:features.indptr[row + 1]]
        for size in (1, 2, 3):
            for terms in combinations(active, size):
                if terms not in rules:
                    continue
                digits = rules[terms][0]
                for position, digit in enumerate(digits):
                    if digit:
                        indicator_values[row, indicator_positions[(position, digit)]] = 1

    names = [f"digit_{position + 1}__{digit}" for position, digit in indicator_pairs]
    return pd.DataFrame(indicator_values, index=index, columns=names)


def predict_cluster_digits(indicators, recorded_codes):
    """Keep known digits and fill X only when all active indicators agree."""
    predictions = []
    for index, recorded in recorded_codes.items():
        predicted = list(recorded)
        for position, digit in enumerate(recorded):
            if digit != "X":
                continue
            candidates = indicators.loc[index].filter(like=f"digit_{position + 1}__")
            active_digits = candidates.index[candidates.eq(1)].str.rsplit("__", n=1).str[-1].tolist()
            if len(active_digits) == 1:
                predicted[position] = active_digits[0]
        predictions.append("".join(predicted))
    return pd.Series(predictions, index=recorded_codes.index, dtype="string")


cluster_digit_reports = []
cluster_digit_rule_tables = {}
cluster_digit_indicator_tables = {}
cluster_digit_test_predictions = {}
cluster_digit_predictions = {}

print("Management codes 0–3 are represented separately. Code 0 includes values filled as missing in the existing preparation.")
print("Age uses fixed decade bands from 18–19 through 110–119, plus 120. Earlier preparation replaces missing or out-of-range age with the complete-record median. These bands are a comparison choice, not an optimized cut-off.")

# Learn the clusters and perfect-alignment rules on development records only.
for preparation, encoder in cluster_development_encoders.items():
    fitting_features, rule_names, rule_fields = cluster_digit_features(encoder, development_matrix)
    test_features, _, _ = cluster_digit_features(encoder, test_matrix)
    rules = fit_cluster_digit_rules(fitting_features, development_targets, rule_fields, min_support=minimum_digit_support)

    # One indicator combines all development rules supporting the same position and digit.
    fitting_indicators = transform_cluster_digit_indicators(fitting_features, rules, rule_fields, development_targets.index)
    test_indicators = transform_cluster_digit_indicators(test_features, rules, rule_fields, test_targets.index)
    blank_codes = pd.Series("XXXX", index=test_targets.index, dtype="string")
    test_codes = predict_cluster_digits(test_indicators, blank_codes)
    cluster_digit_test_predictions[preparation] = pd.DataFrame({"ID": record_ids.loc[test_targets.index], "Recorded occupation": test_targets, "Rule-based prediction": test_codes})
    actual_digits = np.asarray([list(code) for code in test_targets])
    predicted_digits = np.asarray([list(code) for code in test_codes])

    # Coverage and correctness are calculated on the 30 separate complete test records.
    for position in range(4):
        predicted = predicted_digits[:, position] != "X"
        correct = predicted & (predicted_digits[:, position] == actual_digits[:, position])
        active = test_indicators.filter(like=f"digit_{position + 1}__").sum(axis=1)
        cluster_digit_reports.append({"Feature preparation": preparation, "Digit position": position + 1, "Test records": len(test_targets), "Predicted": int(predicted.sum()), "Correct": int(correct.sum()), "Conflicts": int((active > 1).sum()), "Coverage": float(predicted.mean()), "Correct when predicted": float(correct.sum() / predicted.sum()) if predicted.any() else np.nan})

    # Make the observed development rules interpretable by field, digit, and support.
    rule_rows = []
    for terms, (digits, support) in rules.items():
        for position, digit in enumerate(digits):
            if digit:
                rule_rows.append({"Fields": len(terms), "Indicator combination": " + ".join(rule_names[term] for term in terms), "Digit position": position + 1, "Aligned digit": digit, "Development records": support})
    rule_table = pd.DataFrame(rule_rows, columns=["Fields", "Indicator combination", "Digit position", "Aligned digit", "Development records"])
    cluster_digit_rule_tables[preparation] = rule_table
    cluster_digit_indicator_tables[preparation] = {"development": fitting_indicators, "test": test_indicators}

    # Refit the unsupervised clusters and rules on all complete codes for incomplete records.
    if preparation == cluster_best["Feature preparation"]:
        final_encoder = cluster_final_encoder
    else:
        final_encoder = ClusterMembershipFeatures(feature_names=feature_names, feature_positions=text_feature_positions, normalize=cluster_paths[preparation], max_clusters=10, random_state=SEED).fit(training_matrix)
    final_features, final_names, final_fields = cluster_digit_features(final_encoder, training_matrix)
    partial_features, _, _ = cluster_digit_features(final_encoder, prediction_matrix)
    final_rules = fit_cluster_digit_rules(final_features, labelled_targets, final_fields, min_support=minimum_digit_support)
    partial_indicators = transform_cluster_digit_indicators(partial_features, final_rules, final_fields, partial_targets.index)
    partial_predictions = predict_cluster_digits(partial_indicators, partial_targets)
    cluster_digit_indicator_tables[preparation]["incomplete"] = partial_indicators
    cluster_digit_predictions[preparation] = pd.DataFrame({"ID": record_ids.loc[partial_targets.index], "Recorded occupation": partial_targets, "Cluster and survey rule prediction": partial_predictions})

    print(f"\n{preparation}")
    print(f"{len(rules):,} indicator combinations agree on at least one digit in development records, each supported by at least {minimum_digit_support} records.")
    print("Each digit indicator is the union of its agreeing combinations. Conflicting indicators leave the digit unresolved as X.")
    top_rules = rule_table.sort_values("Development records", ascending=False).groupby("Fields", sort=True).head(5)
    display(top_rules.reset_index(drop=True))
    indicator_summary = pd.DataFrame({"Digit indicator": fitting_indicators.columns, "Development records": fitting_indicators.sum(axis=0).to_numpy(), "Test records": test_indicators.sum(axis=0).to_numpy()})
    display(indicator_summary)
    display(cluster_digit_test_predictions[preparation])
    display(cluster_digit_predictions[preparation])

cluster_digit_evaluation = pd.DataFrame(cluster_digit_reports)
print("Test coverage is the share of the 30 complete test records receiving a digit. Correctness is measured only where a digit was proposed.")
print("This is a rule-based digit classifier. Training another classifier on same-row development indicators requires out-of-fold construction.")
print("Both variants are exploratory comparisons. Text preparation was fitted before the split, and 100% development agreement does not establish test accuracy.")
display(cluster_digit_evaluation)


In [ ]:
# Compare text vectors from different records within each original field.
text_values = data.loc[record_ids.index].filter(like="__")
text_names = text_values.columns.to_numpy(dtype=str)
similarity_fields = list(dict.fromkeys(name.split("__", 1)[0] for name in text_names))

# Place the original dot products beside the L2-normalized dot products.
similarity_figure = make_subplots(rows=1, cols=2, subplot_titles=["Raw TF-IDF dot product", "L2-normalized dot product: cosine similarity"], horizontal_spacing=0.18)
similarity_trace_fields = []
similarity_checks = []

for field in similarity_fields:
    # Select this field's token columns and normalize each nonempty record.
    field_values = text_values.loc[:, np.char.startswith(text_names, field + "__")].sparse.to_coo().tocsr()
    field_values.eliminate_zeros()
    unit_values = Normalizer().fit_transform(field_values)

    # Compare each record with every other record in both representations.
    raw_similarity = (field_values @ field_values.T).toarray()
    cosine_similarity = (unit_values @ unit_values.T).toarray()

    # Check the diagonal only where the field has a nonempty vector.
    present = field_values.getnnz(axis=1) > 0
    diagonal = cosine_similarity.diagonal()[present]
    similarity_checks.append({"Text field": field, "TF-IDF columns": field_values.shape[1], "Nonempty records": int(present.sum()), "Minimum normalized self-dot": float(diagonal.min()) if len(diagonal) else np.nan, "Maximum normalized self-dot": float(diagonal.max()) if len(diagonal) else np.nan})

    # Keep one pair of heatmaps per field and show only the selected pair.
    for column, values in enumerate([raw_similarity, cosine_similarity], start=1):
        similarity_figure.add_trace(go.Heatmap(x=record_ids, y=record_ids, z=values, visible=field == similarity_fields[0], colorscale="Teal", colorbar={"title": "Dot product", "x": 0.43 if column == 1 else 1.01}, zmin=0, zmax=1 if column == 2 else None, hovertemplate="Record %{x}<br>Record %{y}<br>Dot product %{z:.3f}<extra></extra>"), row=1, col=column)
        similarity_trace_fields.append(field)

# Let the reader switch fields without calculating the matrices again.
field_buttons = [{"label": field, "method": "update", "args": [{"visible": [name == field for name in similarity_trace_fields]}, {"title": {"text": "Similarity within " + field}}]} for field in similarity_fields]
similarity_figure.update_layout(title={"text": "Similarity within " + similarity_fields[0]}, height=850, margin={"t": 170, "b": 100, "r": 100}, updatemenus=[{"buttons": field_buttons, "x": 0, "y": 1.15, "xanchor": "left", "yanchor": "top"}])
similarity_figure.update_xaxes(title="Record ID")
similarity_figure.update_yaxes(title="Record ID", autorange="reversed")

# Report field sizes and the observed diagonal range for nonempty records.
print("This descriptive display uses no occupation labels. K-means and cluster-count selection use only fitting records in each evaluation round.")
similarity_guide = 'Find records whose text uses similar token weights.\nOn the right, dark off-diagonal cells show greater cosine similarity. A nonempty record matches itself at 1 on the diagonal. Pale cells show little or no overlap.\nBlocks suggest groups with similar wording. Parallel lines depend on record order and do not prove duplicates. The left panel also reflects vector magnitude. Neither panel measures classification accuracy.'
plots.add("Text similarity", similarity_figure, similarity_guide, details=(pd.DataFrame(similarity_checks),))


In [ ]:
# Read the cached validation models without fitting them again.
# ROC measures score discrimination, reliability compares confidence with correctness, and round curves show variation.
diagnostic_rows = ranked_models.copy()
diagnostic_datasets = dataset_options["Dataset"].tolist()
diagnostic_models = list(model_families)
false_positive_grid = np.linspace(0, 1, 101)


def score_validation_estimator(estimator, validation_features, expected):
    """Calculate class-level ROC where defined and confidence reliability for probability outputs."""
    expected = np.asarray(expected)
    classes = np.asarray(estimator.classes_)
    probability_model = hasattr(estimator, "predict_proba")
    scores = estimator.predict_proba(validation_features) if probability_model else estimator.decision_function(validation_features)
    scores = np.asarray(scores)
    if scores.ndim == 1:
        scores = np.column_stack([-scores, scores])

    class_curves = []
    class_areas = []
    positive_counts = []
    for position, occupation_code in enumerate(classes):
        positive = expected == occupation_code
        if positive.any() and not positive.all():
            false_positive, true_positive, _ = roc_curve(positive, scores[:, position])
            interpolated = np.interp(false_positive_grid, false_positive, true_positive)
            interpolated[0] = 0
            interpolated[-1] = 1
            class_curves.append(interpolated)
            class_areas.append(auc(false_positive, true_positive))
            positive_counts.append(int(positive.sum()))

    confidence_rows = []
    if probability_model:
        confidence = scores.max(axis=1)
        correct = classes[scores.argmax(axis=1)] == expected
        confidence_rows = list(zip(confidence, correct))

    return {
        "Curve": np.mean(class_curves, axis=0) if class_curves else None,
        "Macro ROC AUC": float(np.mean(class_areas)) if class_areas else np.nan,
        "ROC-evaluable codes": len(class_areas),
        "ROC codes with one positive validation record": positive_counts.count(1),
        "Validation records": len(expected),
        "Records with codes absent from fitting": int((~np.isin(expected, classes)).sum()),
        "Confidence rows": confidence_rows,
    }


pair_curves = {}
pair_confidence_rows = {}
diagnostic_fold_rows = []
calibration_rows = []

# Collect one already-fitted validation fold for ROC, calibration and stability.
def collect_validation_fold(dataset_name, model_name, fold, estimator, features, expected, macro_f1, accuracy):
    """Record predictions and evaluation coverage without refitting the model."""
    result = score_validation_estimator(estimator, features, expected)
    pair = (dataset_name, model_name)
    if result["Curve"] is not None:
        pair_curves.setdefault(pair, []).append(result["Curve"])
    pair_confidence_rows.setdefault(pair, []).extend(result["Confidence rows"])

    # Preserve the counts needed to interpret ROC when occupation codes are unseen.
    coverage = ["Macro ROC AUC", "ROC-evaluable codes", "ROC codes with one positive validation record", "Validation records", "Records with codes absent from fitting"]
    diagnostic_fold_rows.append({"Feature preparation": dataset_name, "Classifier": model_name, "Round": fold, **{name: result[name] for name in coverage}, "Macro F1": macro_f1, "Accuracy": accuracy})


for candidate_index, configuration in diagnostic_rows.iterrows():
    dataset_name = configuration["Dataset"]
    model_name = configuration["Model"]

    for fold_index, (_, validation_rows) in enumerate(validation_splits):
        # Reuse the model already fitted during the search for this configuration and round.
        configured = clone(model_pipeline).set_params(**search.cv_results_["params"][candidate_index])
        expected = development_targets.iloc[validation_rows]
        cache_key = (joblib.hash(clone(configured)), tuple(expected.index))
        estimator = validation_fit_cache[cache_key]
        # Reuse the scores from model selection for this same validation fold.
        macro_f1 = search.cv_results_[f"split{fold_index}_test_macro_f1"][candidate_index]
        accuracy = search.cv_results_[f"split{fold_index}_test_accuracy"][candidate_index]
        collect_validation_fold(dataset_name, model_name, fold_index + 1, estimator, development_matrix[validation_rows], expected, macro_f1, accuracy)

# Reuse the fitted cluster-compressed validation models produced in the preceding cell.
for (dataset_name, model_name), folds in cluster_diagnostic_folds.items():
    if model_name not in diagnostic_models:
        diagnostic_models.append(model_name)
    if dataset_name not in diagnostic_datasets:
        diagnostic_datasets.append(dataset_name)
    for fold in folds:
        # Cluster models also retain their validation features and scores.
        collect_validation_fold(dataset_name, model_name, fold["fold"], fold["estimator"], fold["features"], fold["expected"], fold["macro_f1"], fold["accuracy"])

roc_curves = {pair: np.mean(curves, axis=0) for pair, curves in pair_curves.items()}
for (dataset_name, model_name), confidence_rows in pair_confidence_rows.items():
    if confidence_rows:
        confidence_frame = pd.DataFrame(confidence_rows, columns=["Confidence", "Correct"])
        confidence_frame["Bin"] = np.minimum((confidence_frame["Confidence"] * 5).astype(int), 4)
        binned = confidence_frame.groupby("Bin", sort=True).agg(**{"Mean confidence": ("Confidence", "mean"), "Observed accuracy": ("Correct", "mean"), "Records": ("Correct", "size")}).reset_index()
        for bin_row in binned.to_dict("records"):
            calibration_rows.append({"Feature preparation": dataset_name, "Classifier": model_name, **bin_row})

diagnostic_fold_results = pd.DataFrame(diagnostic_fold_rows)
calibration_results = pd.DataFrame(calibration_rows)
roc_summary = diagnostic_fold_results.groupby(["Feature preparation", "Classifier"], sort=False).agg(**{
    "Mean ROC AUC": ("Macro ROC AUC", "mean"),
    "ROC AUC SD": ("Macro ROC AUC", lambda values: values.std(ddof=0)),
    "Minimum ROC-evaluable codes": ("ROC-evaluable codes", "min"),
    "Maximum ROC-evaluable codes": ("ROC-evaluable codes", "max"),
}).reset_index()

# Start with the classifier and feature preparation selected by validation F1.
diagnostic_selected_dataset = ranked_models.iloc[0]['Dataset']
diagnostic_selected_model = ranked_models.iloc[0]['Model']
diagnostic_datasets = [diagnostic_selected_dataset] + [name for name in diagnostic_datasets if name != diagnostic_selected_dataset]

# Keep the coverage table for the ROC plot's explanation.
coverage_columns = ['Round', 'Validation records', 'ROC-evaluable codes', 'ROC codes with one positive validation record', 'Records with codes absent from fitting']
roc_coverage = diagnostic_fold_results.loc[diagnostic_fold_results['Feature preparation'].eq(diagnostic_selected_dataset)].groupby('Round', sort=True)[coverage_columns[1:]].first().reset_index()

# Queue all three diagnostics. The final gallery displays them together.
roc_figure, calibration_figure, stability_figure = plots.diagnostics(diagnostic_datasets, diagnostic_models, roc_summary, roc_curves, false_positive_grid, calibration_results, diagnostic_fold_results, diagnostic_selected_dataset, diagnostic_selected_model, roc_coverage)


In [ ]:
learning_guide = ""
# Compare each feature pipeline's leading classifier as the number of training records grows.
learning_started = perf_counter()
learning_candidates = ranked_models.drop_duplicates("Dataset", keep="first")
learning_records = []
learning_failures = []

# Select the same nested fitting groups for every feature pipeline.
def learning_splits():
    """Grow each fitting sample by whole groups while validation stays fixed."""
    for fold, (fit_rows, validation_rows) in enumerate(validation_splits, start=1):
        fitting_groups = development_groups.iloc[fit_rows]
        group_order = fitting_groups.drop_duplicates().sample(frac=1, random_state=SEED + fold)
        group_sizes = fitting_groups.value_counts(sort=False).reindex(group_order)

        for requested_records in [60, 90, 131]:
            group_count = group_sizes.cumsum().searchsorted(requested_records) + 1
            selected_groups = group_order.iloc[:group_count]
            selected_rows = fit_rows[fitting_groups.isin(selected_groups).to_numpy()]
            yield fold, requested_records, selected_rows, validation_rows



for candidate_index, candidate_row in learning_candidates.iterrows():
    candidate_pipeline = clone(model_pipeline).set_params(**search.cv_results_["params"][candidate_index])

    # Reuse the same nested record groups for each classifier.
    for fold, requested_records, selected_rows, validation_rows in learning_splits():
        fitting_targets = development_targets.iloc[selected_rows]
        validation_targets = development_targets.iloc[validation_rows]

        # The full-size endpoint already exists in the validation cache.
        # Only the smaller samples require a new fit to measure the effect of training size.
        try:
            if requested_records == 131:
                cache_key = (joblib.hash(clone(candidate_pipeline)), tuple(validation_targets.index))
                fitted_candidate = validation_fit_cache[cache_key]
            else:
                fitted_candidate = clone(candidate_pipeline).fit(development_matrix[selected_rows], fitting_targets)
        except ValueError as error:
            learning_failures.append({"Feature preparation": candidate_row["Dataset"], "Classifier": candidate_row["Model"], "Fold": fold, "Training records": len(selected_rows), "Reason": str(error)})
            continue

        for partition, rows, targets in [("Training", selected_rows, fitting_targets), ("Validation", validation_rows, validation_targets)]:
            predictions = fitted_candidate.predict(development_matrix[rows])
            learning_records.append({"Feature preparation": candidate_row["Dataset"], "Classifier": candidate_row["Model"], "Requested training records": requested_records, "Training records": len(selected_rows), "Fold": fold, "Partition": partition, "Macro F1": f1_score(targets, predictions, average="macro", zero_division=0), "Accuracy": accuracy_score(targets, predictions), "Training classes": fitting_targets.nunique(), "Validation records with unseen codes": int((~validation_targets.isin(fitting_targets)).sum())})

# Refit each field's clusters inside the smaller training sample for the cluster-compressed paths.
cluster_learning_candidates = cluster_validation_results.sort_values("Macro F1", ascending=False, kind="stable").drop_duplicates("Feature preparation")

for configuration in cluster_learning_candidates.to_dict("records"):
    preparation = configuration["Feature preparation"]
    classifier_name = configuration["Classifier"]
    classifier_parameters = {name.removeprefix("classifier__"): value for name, value in json.loads(configuration["Parameters"]).items()}
    candidate_classifier = clone(cluster_model_families[classifier_name][0]).set_params(**classifier_parameters)

    # Reuse the same nested record groups for each classifier.
    for fold, requested_records, selected_rows, validation_rows in learning_splits():
        fitting_targets = development_targets.iloc[selected_rows]
        validation_targets = development_targets.iloc[validation_rows]

        if requested_records == 131:
            fitting_features, validation_features = cluster_validation_inputs[preparation][fold - 1]
        else:
            field_clusters = ClusterMembershipFeatures(feature_names=feature_names, feature_positions=text_feature_positions, normalize=cluster_paths[preparation], max_clusters=10, random_state=SEED).fit(development_matrix[selected_rows])
            fitting_features = field_clusters.transform(development_matrix[selected_rows])
            validation_features = field_clusters.transform(development_matrix[validation_rows])

        # Reuse the selected full-size cluster model instead of repeating its validation fit.
        try:
            if requested_records == 131:
                fitted_candidate = cluster_diagnostic_folds[(preparation, classifier_name)][fold - 1]["estimator"]
            else:
                fitted_candidate = clone(candidate_classifier).fit(fitting_features, fitting_targets)
        except ValueError as error:
            learning_failures.append({"Feature preparation": preparation, "Classifier": classifier_name, "Fold": fold, "Training records": len(selected_rows), "Reason": str(error)})
            continue

        for partition, features, targets in [("Training", fitting_features, fitting_targets), ("Validation", validation_features, validation_targets)]:
            predictions = fitted_candidate.predict(features)
            learning_records.append({"Feature preparation": preparation, "Classifier": classifier_name, "Requested training records": requested_records, "Training records": len(selected_rows), "Fold": fold, "Partition": partition, "Macro F1": f1_score(targets, predictions, average="macro", zero_division=0), "Accuracy": accuracy_score(targets, predictions), "Training classes": fitting_targets.nunique(), "Validation records with unseen codes": int((~validation_targets.isin(fitting_targets)).sum())})

learning_display_candidates = pd.concat([learning_candidates[["Dataset", "Model"]].rename(columns={"Dataset": "Feature preparation", "Model": "Classifier"}), cluster_learning_candidates[["Feature preparation", "Classifier"]]], ignore_index=True)

# Combine fold scores before computing the mean and variation.
learning_curve_results = pd.DataFrame(learning_records)
learning_curve_failures = pd.DataFrame(learning_failures)
learning_curve_summary = learning_curve_results.groupby(["Feature preparation", "Classifier", "Requested training records", "Partition"], sort=False).agg(**{"Training records": ("Training records", "mean"), "Mean macro-F1": ("Macro F1", "mean"), "Macro-F1 fold SD": ("Macro F1", lambda scores: scores.std(ddof=0)), "Mean accuracy": ("Accuracy", "mean"), "Accuracy fold SD": ("Accuracy", lambda scores: scores.std(ddof=0)), "Rounds": ("Fold", "nunique"), "Training classes": ("Training classes", "mean"), "Validation records with unseen codes": ("Validation records with unseen codes", "mean")}).reset_index()

# Extract the largest available sample from the initially selected curve.
initial_candidate = learning_display_candidates.iloc[0]
initial_curve = learning_curve_summary.loc[(learning_curve_summary["Feature preparation"] == initial_candidate["Feature preparation"]) & (learning_curve_summary["Classifier"] == initial_candidate["Classifier"])]
initial_training = initial_curve.loc[initial_curve["Partition"].eq("Training"), ["Requested training records", "Training records", "Mean macro-F1"]]
initial_validation = initial_curve.loc[initial_curve["Partition"].eq("Validation"), ["Requested training records", "Mean macro-F1"]]
initial_scores = initial_training.merge(initial_validation, on="Requested training records", suffixes=(" training", " validation")).sort_values("Requested training records")

# Explain the lines with their actual initial scores and a cautious interpretation.
example = "No paired training and validation scores are available."
if not initial_scores.empty:
    last_score = initial_scores.iloc[-1]
    example = f"In the initial view at {last_score['Training records']:.0f} average training records, macro F1 is {last_score['Mean macro-F1 training']:.2f} for training and {last_score['Mean macro-F1 validation']:.2f} for validation."
learning_guide += "\n\n" + f"**Learning curve:** The solid line shows training and the dashed line shows validation. {example} A large gap suggests overfitting. Both lines low suggests underfitting. Rising validation scores suggest more data may help. Error bars show fold variation."
if not learning_curve_failures.empty:
    learning_guide += "\n\n" + "Some smaller training sets could not fit the selected feature pipeline. Their curves omit those fits and report the reasons below."

# Build one chart with a selector for each pipeline and score.
learning_figure = go.Figure()
learning_options = []

for preparation, classifier in learning_display_candidates.itertuples(index=False, name=None):
    selected_curve = learning_curve_summary.loc[learning_curve_summary["Feature preparation"].eq(preparation)]

    for metric, spread in [("Mean macro-F1", "Macro-F1 fold SD"), ("Mean accuracy", "Accuracy fold SD")]:
        trace_start = len(learning_figure.data)
        display_preparation = preparation.replace("; ", ";<br>").replace(". ", ".<br>")
        option_title = f"{display_preparation}<br>{classifier} · {metric}"

        # Let Plotly build the two labeled lines and their fold-variation bars.
        values = selected_curve.sort_values("Requested training records")
        curve = px.line(values, x="Training records", y=metric, color="Partition", line_dash="Partition", error_y=spread, markers=True, hover_data={"Training records": ":.0f", metric: ":.3f", "Rounds": ":.0f", "Training classes": ":.1f", "Validation records with unseen codes": ":.1f"}, category_orders={"Partition": ["Training", "Validation"]}, color_discrete_map={"Training": "#234737", "Validation": "#507589"}, line_dash_map={"Training": "solid", "Validation": "dash"})
        for trace in curve.data:
            trace.update(visible=trace_start == 0, marker_size=9)
            learning_figure.add_trace(trace)

        learning_options.append((trace_start, option_title, metric))

# Select the training and validation traces for the chosen curve.
learning_buttons = []
for trace_start, title, metric in learning_options:
    visible = [trace in [trace_start, trace_start + 1] for trace in range(len(learning_figure.data))]
    learning_buttons.append({"label": title, "method": "update", "args": [{"visible": visible}, {"title": {"text": f"Learning curve · {metric}"}, "yaxis": {"title": metric, "range": [0, 1.05]}}]})

# Make the explanatory title, controls, axes, and legend readable in Colab.
learning_figure.update_layout(width=1000, height=560, margin={"t": 220, "b": 85})
learning_figure.update_layout(title={"text": "Learning curve · Mean macro-F1", "font": {"size": 20}}, xaxis={"title": "Actual training records", "rangemode": "tozero"}, yaxis={"title": "Mean macro-F1", "range": [0, 1.05]})
learning_figure.update_layout(updatemenus=[{"buttons": learning_buttons, "direction": "down", "x": 0, "y": 1.50, "xanchor": "left", "yanchor": "top", "font": {"size": 14}}], legend={"orientation": "h", "y": -0.2, "font": {"size": 16}})
plots.add("Learning curve", learning_figure, learning_guide, details=(learning_curve_failures,) if not learning_curve_failures.empty else ())
print(f"Learning curves calculated in {(perf_counter() - learning_started) / 60:.2f} minutes.")



In [ ]:
# Compare the existing validation results without fitting the models again.
model_scores = ranked_models.rename(columns={"Dataset": "Feature pipeline", "Model": "Classifier"})
cluster_scores = cluster_validation_results.rename(columns={"Feature preparation": "Feature pipeline"})
representation_results = pd.concat([model_scores, cluster_scores], ignore_index=True)
representation_order = dataset_options["Dataset"].tolist() + list(cluster_paths)
classifier_order = list(model_families)

# Rank pipelines using only classifiers evaluated for every pipeline.
classifier_coverage = representation_results.groupby("Classifier")["Feature pipeline"].nunique()
common_classifiers = classifier_coverage.index[classifier_coverage.eq(len(representation_order))]
shared_scores = representation_results.loc[representation_results["Classifier"].isin(common_classifiers)]
pipeline_behavior = shared_scores.groupby("Feature pipeline")["Macro F1"].agg(["mean", "min", "max", "std"])

if not pipeline_behavior.empty:
    representation_order = pipeline_behavior.sort_values("mean", ascending=False, kind="stable").index.tolist()
    ranking_description = f"Ranked by mean macro F1 across {len(common_classifiers)} shared classifiers"
else:
    ranking_description = "Original order: no classifier was evaluated on every pipeline"

# Name each pipeline briefly. The complete preparation remains available on hover.
short_names = {}
for dataset in dataset_options.to_dict("records"):
    steps = []
    for option, description in [("L2", "L2 normalization"), ("Digit selection", "digit-aligned selection"), ("SVD", "SVD")]:
        if dataset[option]:
            steps.append(description)
    short_names[dataset["Dataset"]] = "Text encoded with TF-IDF<br>Non-text fields prepared separately" + ("<br>Then " + ", then ".join(steps) if steps else "")

# Cluster paths normalize each text field only when their name says so.
for preparation, normalize in cluster_paths.items():
    text = "L2-normalized TF-IDF" if normalize else "TF-IDF without L2 normalization"
    short_names[preparation] = f"Field clusters from {text}<br>Non-text fields prepared separately"

# Display rank only when every pipeline shares classifiers for comparison.
pipeline_labels = [f"{rank}. {short_names[name]}" if not pipeline_behavior.empty else short_names[name] for rank, name in enumerate(representation_order, start=1)]
classifier_labels = [name.replace("Complement Naive Bayes", "Complement<br>Naive Bayes").replace("Or balanced occupation ensemble", "Or balanced<br>occupation ensemble") for name in classifier_order]
pipeline_names = np.asarray([[name] * len(classifier_order) for name in representation_order])
fold_variation = representation_results.pivot(index="Feature pipeline", columns="Classifier", values="Fold SD").reindex(index=representation_order, columns=classifier_order).to_numpy()

# Prepare one table for each selectable validation metric.
score_views = {
    "Mean validation macro F1": representation_results.pivot(index="Feature pipeline", columns="Classifier", values="Macro F1"),
    "Mean validation accuracy": representation_results.pivot(index="Feature pipeline", columns="Classifier", values="Accuracy"),
    "Mean validation ROC AUC for evaluable codes": roc_summary.pivot(index="Feature preparation", columns="Classifier", values="Mean ROC AUC"),
}
comparison_figure = go.Figure()
score_colors = [[0, "#f5aaa2"], [0.5, "#fff1b5"], [1, "#80cba7"]]

for number, (metric, table) in enumerate(score_views.items()):
    # Color compares the scores in this metric; the numbers show the absolute scores.
    values = table.reindex(index=representation_order, columns=classifier_order).to_numpy()
    available = values[np.isfinite(values)]
    lower = float(available.min()) if available.size else 0
    upper = float(available.max()) if available.size else 1
    upper = upper if upper > lower else lower + 1
    hover = "%{customdata[0]}<br>Classifier: %{x}<br>Score: %{z:.3f}"
    if number == 0:
        hover += "<br>Macro F1 fold SD: %{customdata[1]}"

    # A dash marks an unavailable result rather than displaying it as zero.
    comparison_figure.add_trace(go.Heatmap(x=classifier_order, y=pipeline_labels, z=values, zmin=lower, zmax=upper, colorscale=score_colors, texttemplate="%{z:.2f}", textfont={"size": 17, "color": "#172f26"}, customdata=np.stack([pipeline_names, fold_variation.astype(str)], axis=-1), hovertemplate=hover + "<extra></extra>", hoverongaps=False, colorbar={"title": {"text": "Score<br>relative colors", "font": {"size": 17}}, "tickfont": {"size": 17}, "tickformat": ".2f"}, xgap=5, ygap=5, visible=number == 0))
    missing_rows, missing_columns = np.where(~np.isfinite(values))
    comparison_figure.add_trace(go.Scatter(x=[classifier_order[col] for col in missing_columns], y=[pipeline_labels[row] for row in missing_rows], mode="text", text=["—"] * len(missing_rows), textfont={"size": 20, "color": "#667a72"}, showlegend=False, hovertemplate="Not available or not evaluated<extra></extra>", visible=number == 0))

# Switch the metric while keeping the pipeline order fixed.
score_buttons = [{"label": title, "method": "update", "args": [{"visible": [index // 2 == selected for index in range(2 * len(score_views))]}, {"title": {"text": f"{title}<br><sup>{ranking_description}</sup>"}}]} for selected, title in enumerate(score_views)]
comparison_figure.update_layout(title=f"Mean validation macro F1<br><sup>{ranking_description}</sup>", plot_bgcolor="#eef2f0", height=max(1100, 95 * len(representation_order) + 200), margin={"t": 190, "b": 70, "r": 100}, xaxis={"side": "top", "tickmode": "array", "tickvals": classifier_order, "ticktext": classifier_labels, "tickangle": 0, "tickfont": {"size": 17}, "automargin": True, "showgrid": False}, yaxis={"autorange": "reversed", "automargin": True, "tickfont": {"size": 15}, "showgrid": False}, updatemenus=[{"buttons": score_buttons, "font": {"size": 17}, "x": 0, "y": 1.07, "xanchor": "left", "yanchor": "top"}])

# Explain how to read the comparison before displaying it.
guidance = 'Choose a preparation that works consistently across classifiers by reading each complete row.\nGreen means a higher score and red a lower score within the selected metric. Read the printed numbers for actual scores. A dash means unavailable.\nText uses TF-IDF. Categorical survey fields are one-hot encoded separately, then combined. L2, feature selection and SVD operate afterward. Cluster paths one-hot encode cluster assignments. Hover shows full details.'
plots.add("Feature preparation across classifiers", comparison_figure, guidance)


In [ ]:
# Render every registered graph and its reading guide in order.
plots.show()
